# Unknown Dataset Hackathon — Google Colab

**GitHub repository:** https://github.com/DimitriAntypas/unknown-dataset-hackathon

**Team:** Georgio Sleiman, Andrew Sleiman, Carla Doumit, Dimitri Antypas

This notebook contains the corrected Python program. No GitHub repository or separate code ZIP is needed to run it.
Use **Runtime → Change runtime type → CPU**, then run cells in order with the ▶ buttons.

**Steps 1–3 run a complete demonstration. Steps 4–6 run your uploaded CSV data.**
Outputs: predictions.csv and profile.json. The program handles classification, regression and clustering.
The official assessment format remains unverified. Colab hardware and timing differ from the assessment CPU.


In [ ]:
# @title 1. Set up the program (first run downloads dependencies)
import json, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
if sys.version_info < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11 or newer.")
BASE = Path('/content') if Path('/content').exists() else Path.cwd()
PROJECT = BASE / 'udh_colab'
PROJECT.mkdir(exist_ok=True)
SOURCES = {'validate_real.py': '"""Optional validation on the three public example CSVs; no network downloads.\nUsage: python validate_real.py --data-dir /path/to/extracted/csvs\nThese are examples, not the official nine practice datasets or official scores.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--data-dir\', required=True)\n    ap.add_argument(\'--report\', default=\'real_results.json\')\n    a = ap.parse_args()\n    root = Path(__file__).resolve().parent\n    cases = [(\'Telco\', \'WA_Fn-UseC_-Telco-Customer-Churn.csv\', \'Churn\', \'customerID\'),\n             (\'Maintenance\',\'ai4i2020.csv\',\'Machine failure\',\'UDI\'),\n             (\'Mall\',\'Mall_Customers.csv\',None,\'CustomerID\')]\n    results = []\n    for name, filename, target, idcol in cases:\n        matches = list(Path(a.data_dir).rglob(filename))\n        if len(matches) != 1:\n            raise ValueError(f\'Expected exactly one {filename}, found {len(matches)}\')\n        df = pd.read_csv(matches[0])\n        if name == \'Maintenance\':\n            # Failure indicators reveal the target; do not use them as predictors.\n            df = df.drop(columns=[\'TWF\',\'HDF\',\'PWF\',\'OSF\',\'RNF\',\'Product ID\'], errors=\'ignore\')\n        with tempfile.TemporaryDirectory() as tmp:\n            t = Path(tmp)\n            command = [sys.executable, str(root/\'main.py\'), \'--out-dir\',str(t/\'out\'),\'--id-col\',idcol]\n            truth = None\n            if target:\n                tr, te = train_test_split(df, test_size=.25, random_state=42, stratify=df[target])\n                truth = te[target].reset_index(drop=True)\n                tr.to_csv(t/\'train.csv\',index=False)\n                te.drop(columns=target).to_csv(t/\'test.csv\',index=False)\n                command += [\'--train\',str(t/\'train.csv\'),\'--test\',str(t/\'test.csv\'),\'--target\',target,\'--task\',\'classification\']\n            else:\n                df.to_csv(t/\'train.csv\',index=False)\n                command += [\'--train\',str(t/\'train.csv\'),\'--task\',\'clustering\',\'--n-clusters\',\'5\']\n            start = time.monotonic()\n            proc = subprocess.run(command, capture_output=True, text=True, timeout=60)\n            if proc.returncode:\n                raise RuntimeError(proc.stderr)\n            wall = time.monotonic()-start\n            predictions = pd.read_csv(t/\'out\'/\'predictions.csv\')\n            profile = json.loads((t/\'out\'/\'profile.json\').read_text())\n            expected_ids = te[idcol].to_numpy() if target else df[idcol].to_numpy()\n            assert (predictions[idcol].to_numpy() == expected_ids).all()\n            result = {\'dataset\':name,\'wall_seconds\':round(wall,3),\n                      \'models\':profile[\'model_report\'].get(\'chosen_models\'),\n                      \'timeout\':profile[\'model_report\'][\'supervisor_timeout\']}\n            if target:\n                result.update(accuracy=accuracy_score(truth,predictions.prediction),\n                    balanced_accuracy=balanced_accuracy_score(truth,predictions.prediction),\n                    macro_f1=f1_score(truth,predictions.prediction,average=\'macro\'))\n                positive = \'Yes\' if name == \'Telco\' else 1\n                result[\'roc_auc\'] = roc_auc_score(truth == positive,predictions[\'proba_\'+str(positive)])\n            else:\n                assert predictions.cluster.nunique() == 5\n                result[\'clusters\'] = 5\n                result[\'note\'] = \'Five clusters assumed for this example; no ground-truth labels supplied.\'\n            results.append(result)\n            print(json.dumps(result), flush=True)\n    Path(a.report).write_text(json.dumps(results,indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'START_HERE.md': "# Start here\n\n1. Extract the ZIP and open the folder containing `main.py` and `requirements.txt`.\n2. Install Python 3.12. In a terminal inside that folder run:\n\n   ```bash\n   python -m venv .venv\n   ```\n\n   Activate it on Windows PowerShell with `.venv\\Scripts\\Activate.ps1`,\n   or on macOS/Linux with `source .venv/bin/activate`.\n\n3. Install and check:\n\n   ```bash\n   pip install -r requirements.txt\n   python tests/test_pipeline.py\n   python tests/test_fixes.py\n   python tests/test_review_fixes.py\n   python make_synthetic.py\n   python evaluate.py\n   ```\n\n4. Upload or update the extracted project files in https://github.com/DimitriAntypas/unknown-dataset-hackathon.\n   Keep `main.py`, `supervisor.py`, `requirements.txt`, `README.md`, all three test files,\n   and the other included source files together. Include `.github/workflows/ci.yml`.\n   Make the repository public. Do not upload virtual environments, generated outputs,\n   credentials, or the downloaded Kaggle datasets.\n5. All four supplied team names are retained. Confirm an exception to the brief's three-student team size\n   before submitting, or update the names to match the approved team.\n6. At kickoff, use the professor's exact command, metadata and output specification.\n   Adapt this code if necessary and run the official checker on all nine official practice datasets.\n   The package currently assumes the command-line and CSV/profile formats documented in README.md.\n7. Submit https://github.com/DimitriAntypas/unknown-dataset-hackathon according to the professor's deadline instructions.\n\nNo website is required. The existing GitHub instructions PDF is a general guide;\nthis file and README.md describe the revised package, including its new supervisor module.\n\nThe supplied public datasets are optional additional tests. Extract their CSVs and run:\n\n```bash\npython validate_real.py --data-dir /path/to/csv-folder\n```\n\nThe program does not download data or contact online services when it runs.\n", 'supervisor.py': '"""Portable CLI wall-clock supervision; no scientific imports in the parent."""\nimport argparse\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\n\ndef run(argv):\n    parser = argparse.ArgumentParser(add_help=False)\n    parser.add_argument(\'--out-dir\', default=\'out\')\n    parser.add_argument(\'--time-budget\', type=float, default=40)\n    opts, _ = parser.parse_known_args(argv)\n    if not math.isfinite(opts.time_budget) or opts.time_budget <= 0:\n        parser.error(\'--time-budget must be finite and positive\')\n    if \'--help\' in argv or \'-h\' in argv:\n        return subprocess.call([sys.executable, str(Path(__file__).with_name(\'main.py\')), \'--_worker\', *argv])\n    started = time.monotonic()\n    # Reserve at least eight seconds of the contest\'s 60-second allowance for\n    # parent startup, termination and publishing; this is not a hardware guarantee.\n    deadline = min(52.0, opts.time_budget + 8.0)\n    out = Path(opts.out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=\'udh-worker-\') as scratch:\n        env = os.environ.copy()\n        for key in (\'OMP_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'MKL_NUM_THREADS\'):\n            env[key] = \'1\'\n        command = [sys.executable, str(Path(__file__).with_name(\'main.py\')), \'--_worker\', *argv, \'--out-dir\', scratch]\n        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=env)\n        timed_out = False\n        try:\n            stdout, stderr = process.communicate(timeout=deadline)\n        except subprocess.TimeoutExpired:\n            timed_out = True\n            process.kill()\n            stdout, stderr = process.communicate()\n        root = Path(scratch)\n        if timed_out:\n            marker = root / \'checkpoint.json\'\n            if not marker.exists():\n                print(\'Time limit reached before a usable checkpoint was ready.\', file=sys.stderr)\n                return 124\n            source = root / json.loads(marker.read_text())[\'directory\']\n        elif process.returncode:\n            print(stderr or stdout, file=sys.stderr)\n            return process.returncode\n        else:\n            source = root\n        profile = json.loads((source / \'profile.json\').read_text())\n        report = profile[\'model_report\']\n        report[\'supervisor_timeout\'] = timed_out\n        report[\'wall_seconds\'] = round(time.monotonic() - started, 3)\n        for name in (\'predictions.csv\', \'profile.json\'):\n            temporary = out / (name + \'.tmp\')\n            if name == \'profile.json\':\n                temporary.write_text(json.dumps(profile, indent=2))\n            else:\n                shutil.copyfile(source / name, temporary)\n            os.replace(temporary, out / name)\n        print(json.dumps({\'task\': profile[\'task\'], \'wall_seconds\': report[\'wall_seconds\'],\n                          \'timeout_fallback\': timed_out, \'models\': report.get(\'chosen_models\')}))\n        return 0\n', 'evaluate.py': '"""Local benchmark: quality + runtime for every table in synth/ (run make_synthetic.py first)."""\nimport json, subprocess, sys, time\nimport numpy as np, pandas as pd\nfrom sklearn.metrics import (accuracy_score, adjusted_rand_score, balanced_accuracy_score, f1_score, r2_score)\n\nrows = []\nfailures = []\nfor m in json.load(open("synth/manifest.json")):\n    n, task = m["name"], m["task"]\n    t = time.time()\n    if task == "clustering":\n        args = ["--train", f"synth/{n}.csv", "--task", "clustering", "--n-clusters", str(m["k"]), "--id-col", "row_id"]\n    else:\n        args = ["--train", f"synth/{n}_train.csv", "--test", f"synth/{n}_test.csv", "--target", m.get("target", "target"),\n                "--task", task]\n        if n != "bike":\n            args += ["--id-col", "row_id"]\n    try:\n        r = subprocess.run([sys.executable, "main.py", *args, "--out-dir", f"out/{n}"], capture_output=True, text=True, timeout=60)\n    except subprocess.TimeoutExpired:\n        failures.append(n)\n        print(n, "FAILED: exceeded 60 seconds")\n        continue\n    wall = time.time() - t\n    if r.returncode:\n        failures.append(n); print(n, "FAILED\\n", r.stderr[-1500:]); continue\n    p = pd.read_csv(f"out/{n}/predictions.csv")\n    if task == "clustering":\n        y = pd.read_csv(f"synth/{n}_truth.csv")["y"]; rows.append((n, task, f"ARI={adjusted_rand_score(y, p[\'cluster\']):.3f}", wall))\n    else:\n        y = pd.read_csv(f"synth/{n}_truth.csv")["target"]; pr = p["prediction"]\n        if task == "classification":\n            s = f"acc={accuracy_score(y, pr):.3f} bacc={balanced_accuracy_score(y, pr):.3f} f1m={f1_score(y, pr, average=\'macro\'):.3f}"\n        else:\n            s = f"R2={r2_score(y, pr):.3f}"\n        rows.append((n, task, s, wall))\nfor r in rows:\n    print(f"{r[0]:8s} {r[1]:15s} {r[2]:40s} {r[3]:5.1f}s")\nprint("max wall:", round(max((r[3] for r in rows), default=0), 1), "s")\n\nif failures:\n    raise SystemExit("Failed datasets: " + ", ".join(failures))\n', 'VALIDATION.md': '# Validation of the corrected submission\n\nValidated on 24 October 2026. All four defects found in the submission review are corrected in main.py and in the self-contained Colab notebook.\n\n## Corrections and regression checks\n\n- Sorted integer measurements are preserved. The new end-to-end test requires at least 95% accuracy on unseen rows where a sorted integer measurement determines the target. Explicitly named ID columns are still removed.\n- Boolean training columns accept native booleans, text forms of True and False, numeric 0 and 1, and missing markers in test data. An end-to-end test verifies successful output and the predictions for known boolean values.\n- Delimiter detection only considers comma, semicolon, tab and pipe. Tests preserve single-column headers and values, quoted commas inside a single column, and all supported separators.\n- Missing-value counts, fractions, cell totals and row totals are calculated from one normalized mask. Tests check datetime question marks and whitespace, numeric-text nulls and markers, nullable booleans, and overlapping missing cells in the same row.\n\n## Test execution and environment\n\nAll 19 checks passed: seven original pipeline tests, four original robustness tests and eight new regression checks. They were run as the following scripts, without requiring pytest:\n\n    python tests/test_pipeline.py\n    python tests/test_fixes.py\n    python tests/test_review_fixes.py\n\nThe deliberately stalled worker was terminated and its saved checkpoint recovered after approximately 8.02 seconds. Notebook code cells parse successfully and their embedded source files match the supplied project files.\n\nValidation used Python 3.12.14, numpy 2.3.5, pandas 2.2.3, scikit-learn 1.8.0, scipy 1.17.0.\nThis was the available runtime, not a fresh installation of the exact versions in requirements.txt. The dependency pins are preserved.\n\n## Synthetic benchmark results\n\nAll eleven synthetic benchmarks completed successfully without supervisor timeout or fallback. Runtime measures the whole CLI process on this machine and is rounded to 0.1 seconds. These are local examples rather than official contest scores.\n\n| Dataset | Task | Metric | Value | Wall seconds |\n|---|---|---|---:|---:|\n| bc | classification | accuracy | 0.952 | 4.5 |\n| wine | classification | accuracy | 0.944 | 4.2 |\n| imb | classification | accuracy | 0.940 | 7.0 |\n| moons | classification | accuracy | 0.923 | 4.9 |\n| diab | regression | r2 | 0.492 | 5.8 |\n| fried | regression | r2 | 0.898 | 15.9 |\n| bike | regression | r2 | 0.958 | 12.1 |\n| iris | clustering | ari | 0.886 | 1.7 |\n| blobs | clustering | ari | 0.997 | 2.2 |\n| moons_c | clustering | ari | 0.496 | 1.5 |\n| blobs5 | clustering | ari | 1.000 | 2.5 |\n\nMachine-readable environment, test counts, benchmark metrics and model selections are included in validation_results.json. Reproduce the benchmarks with:\n\n    python make_synthetic.py\n    python evaluate.py\n\n## Remaining submission requirements and limitations\n\n- The submission repository URL is https://github.com/DimitriAntypas/unknown-dataset-hackathon. The README includes this URL. A Colab link is optional unless separately requested.\n- The supplied four team names are preserved. The brief specifies three students per team, so confirm an exception or update the team before submission.\n- Run the official checker after adapting the entry command, metadata and output schema to the kickoff specification. Those materials and the nine official practice datasets were not supplied.\n- Strongly non-convex clustering remains a performance limitation; the noisy two-moons result is reported above.\n- Local timing does not certify timing on the assessment CPU or larger tables. The supervisor and checkpoint fallback remain in place.\n- Earlier public-example measurements are retained in real_results.json as historical results from the uploaded package. Those CSVs were not included in this ZIP and were not rerun for this correction.\n', 'requirements.txt': 'numpy==2.4.4\npandas==3.0.2\nscikit-learn==1.8.0\nscipy==1.17.1\n', 'main.py': '#!/usr/bin/env python3\n"""Unknown Dataset Hackathon - automatic machine learning on unfamiliar tables.\n\nSupervised:  python main.py --train train.csv --test test.csv --target y --out-dir out\nClustering:  python main.py --train data.csv --task clustering --n-clusters 4 --out-dir out\n\nWrites  <out-dir>/predictions.csv  and  <out-dir>/profile.json.\nAdapt CLI arguments, metadata parsing, profile fields and I/O if the official contract differs.\n"""\n# Start the CLI supervisor before importing numerical libraries.\nif __name__ == "__main__":\n    import sys\n    if "--_worker" not in sys.argv:\n        from supervisor import run\n        raise SystemExit(run(sys.argv[1:]))\n    sys.argv.remove("--_worker")\n\nimport argparse\nimport csv\nimport json\nimport os\nimport time\nimport warnings\n\n# Respect an explicitly configured thread limit; otherwise use one CPU thread.\nfor _key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ.setdefault(_key, "1")\nfrom collections import Counter, OrderedDict\n\nimport numpy as np\nimport pandas as pd\n\nwarnings.filterwarnings("ignore")\n\nfrom scipy.stats import skew as _skew\nfrom sklearn.base import BaseEstimator, TransformerMixin, clone\nfrom sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering\nfrom sklearn.compose import ColumnTransformer, TransformedTargetRegressor\nfrom sklearn.decomposition import PCA\nfrom sklearn.ensemble import (ExtraTreesClassifier, ExtraTreesRegressor,\n                              HistGradientBoostingClassifier, HistGradientBoostingRegressor,\n                              RandomForestClassifier, RandomForestRegressor)\nfrom sklearn.feature_selection import mutual_info_classif, mutual_info_regression\nfrom sklearn.experimental import enable_iterative_imputer  # noqa: F401\nfrom sklearn.impute import IterativeImputer, SimpleImputer\nfrom sklearn.linear_model import LogisticRegression, RidgeCV\nfrom sklearn.metrics import (accuracy_score, adjusted_rand_score, balanced_accuracy_score,\n                             f1_score, r2_score, silhouette_score)\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.model_selection import KFold, StratifiedKFold\nfrom sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor, NearestNeighbors\nfrom sklearn.neural_network import MLPClassifier, MLPRegressor\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import (LabelEncoder, OneHotEncoder, PolynomialFeatures,\n                                   StandardScaler)\n\nSEED = 42\nSTART = time.monotonic()  # legacy import timestamp; budgets are per invocation\nMISSING_TOKEN = "__missing__"\n\n\n# =============================================================================== time budget\nclass Budget:\n    """Wall-clock guard. The contest limit is 60 s per dataset; we plan for `total` (default 40 s)."""\n\n    def __init__(self, total=40.0):\n        self.total = float(total)\n        if not np.isfinite(self.total) or self.total <= 0:\n            raise ValueError("time_budget must be finite and positive")\n        self.started = time.monotonic()\n\n    def elapsed(self):\n        return time.monotonic() - self.started\n\n    def left(self):\n        return self.total - self.elapsed()\n\n    def select_open(self):  # model search stops after ~half the budget\n        return self.elapsed() < 0.5 * self.total\n\n    def select_hard_stop(self):  # abort a candidate that is still running after ~70 %\n        return self.elapsed() > 0.7 * self.total\n\n\n# =============================================================================== feature cleaning\nclass FeatureCleaner:\n    """Learns column types and a cleaning plan on training features; applies it consistently to any table.\n\n    Handles: numbers stored as text, datetime strings, booleans, blank/`?` missing markers, infinities,\n    constant / duplicate / perfectly-correlated / ID-like columns, high-cardinality categoricals (frequency\n    encoding), and test tables with missing or extra columns.\n    """\n    DT_PARTS = ("year", "month", "day", "dayofweek", "hour", "epoch_days")\n\n    # ---- type detection\n    @staticmethod\n    def _detect(s):\n        if pd.api.types.is_bool_dtype(s):\n            return "bool"\n        if pd.api.types.is_datetime64_any_dtype(s):\n            return "datetime"\n        if pd.api.types.is_numeric_dtype(s):\n            return "numeric"\n        ss = s.dropna().astype(str).str.strip()\n        ss = ss[ss != ""]\n        if len(ss) == 0:\n            return "categorical"\n        if pd.to_numeric(ss, errors="coerce").notna().mean() > 0.95:\n            return "numeric_text"\n        sample = ss.sample(min(len(ss), 200), random_state=SEED)\n        if sample.str.contains(r"\\d").mean() > 0.9 and sample.str.len().median() >= 6:\n            try:\n                if pd.to_datetime(sample, errors="coerce").notna().mean() > 0.95:\n                    return "datetime"\n            except Exception:\n                pass\n        return "categorical"\n\n    def _basic(self, X, fit):\n        out = OrderedDict()\n        for c in self.columns_:\n            s = X[c] if c in X.columns else pd.Series(np.nan, index=X.index)\n            kind = self.kinds_[c]\n            if kind == "datetime":\n                dt = pd.to_datetime(s, errors="coerce")\n                try:\n                    dt = dt.dt.tz_localize(None)\n                except Exception:\n                    pass\n                parts = {"year": dt.dt.year, "month": dt.dt.month, "day": dt.dt.day,\n                         "dayofweek": dt.dt.dayofweek, "hour": dt.dt.hour,\n                         "epoch_days": (dt - pd.Timestamp("1970-01-01")).dt.total_seconds() / 86400.0}\n                if fit:\n                    self.dt_keep_[c] = [p for p in self.DT_PARTS if parts[p].nunique(dropna=True) > 1]\n                for p in self.dt_keep_[c]:\n                    out[f"{c}__{p}"] = parts[p].astype(float)\n            elif kind == "bool":\n                # CSV readers may return text when a test column has missing\n                # markers even though its training counterpart is boolean.\n                values = s.astype("string").str.strip().str.casefold()\n                out[c] = values.map({\n                    "true": 1.0, "false": 0.0, "1": 1.0, "0": 0.0,\n                    "1.0": 1.0, "0.0": 0.0, "yes": 1.0, "no": 0.0,\n                }).astype(float)\n            elif kind in ("numeric", "numeric_text"):\n                out[c] = pd.to_numeric(s, errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)\n            else:\n                t = s.astype("string").str.strip()\n                t = t.mask(t.isin(["", "?"]))\n                out[c] = t.astype(object).where(t.notna(), np.nan)\n        return pd.DataFrame(out, index=X.index)\n\n    # ---- fit / transform\n    def fit(self, X):\n        self.columns_ = list(X.columns)\n        self.kinds_ = {c: self._detect(X[c]) for c in X.columns}\n        self.dt_keep_ = {}\n        basic = self._basic(X, fit=True)\n        n = len(basic)\n        drops = OrderedDict()\n        self.dup_pairs_, self.near_pairs_ = [], []\n        is_num = {c: pd.api.types.is_numeric_dtype(basic[c]) for c in basic.columns}\n\n        for c in basic.columns:  # constant / empty\n            if basic[c].nunique(dropna=True) <= 1:\n                drops[c] = "constant"\n        for c in basic.columns:  # ID-like\n            if c in drops:\n                continue\n            s = basic[c]\n            nun = s.nunique(dropna=True)\n            if not is_num[c]:\n                if n > 20 and nun > 0.9 * n:\n                    drops[c] = "id_like"\n            else:\n                raw_name = c.split("__")[0]\n                name_idish = raw_name.lower() in {"id", "index", "idx", "rowid", "row_id", "unnamed: 0", "customerid"}\n                intlike = s.notna().all() and np.allclose(s, np.round(s))\n                # A sorted integer measurement can carry the entire predictive\n                # signal. Require an ID-like name instead of relying on order.\n                if intlike and nun == n and n > 20 and name_idish:\n                    drops[c] = "id_like"\n        seen = {}\n        for c in basic.columns:  # exact duplicates\n            if c in drops:\n                continue\n            s = basic[c]\n            key = (is_num[c], int(pd.util.hash_pandas_object(s.astype(str), index=False).sum() % (2 ** 61)))\n            if key in seen and s.astype(str).equals(basic[seen[key]].astype(str)):\n                drops[c] = f"duplicate_of:{seen[key]}"\n                self.dup_pairs_.append([seen[key], c])\n            else:\n                seen.setdefault(key, c)\n        num_left = [c for c in basic.columns if c not in drops and is_num[c]]\n        if 2 <= len(num_left) <= 400:  # perfectly correlated (scaled / shifted copies)\n            M = basic[num_left].to_numpy(float)\n            med = np.nanmedian(M, axis=0)\n            M = np.where(np.isnan(M), med, M)\n            sd = M.std(0)\n            ok = sd > 1e-12\n            Z = (M - M.mean(0)) / np.where(ok, sd, 1)\n            C = np.abs(Z.T @ Z / n)\n            for j in range(len(num_left)):\n                if num_left[j] in drops:\n                    continue\n                for i in range(j):\n                    if num_left[i] not in drops and ok[i] and ok[j] and C[i, j] > 0.9999:\n                        drops[num_left[j]] = f"perfectly_correlated_with:{num_left[i]}"\n                        self.near_pairs_.append([num_left[i], num_left[j]])\n                        break\n        self.dropped_ = drops\n        self.out_columns_ = [c for c in basic.columns if c not in drops]\n        self.freq_maps_ = {}\n        for c in self.out_columns_:\n            if not is_num[c] and basic[c].nunique() > 20:\n                self.freq_maps_[c] = basic[c].value_counts(normalize=True).to_dict()\n        self.basic_ = basic  # kept for profiling; released by the caller\n        self.is_num_ = is_num\n        return self\n\n    def transform(self, X):\n        basic = self._basic(X, fit=False)\n        out = basic[self.out_columns_].copy()\n        for c, m in self.freq_maps_.items():\n            out[f"{c}__freq"] = basic[c].map(m).fillna(0.0).astype(float)\n        return out\n\n\n# =============================================================================== model preprocessing\nclass SkewFix(BaseEstimator, TransformerMixin):\n    """Signed log1p on strongly skewed columns (learned on training data)."""\n\n    def __init__(self, thr=1.5):\n        self.thr = thr\n\n    def fit(self, X, y=None):\n        X = np.asarray(X, float)\n        with np.errstate(all="ignore"):\n            sk = _skew(X, axis=0, nan_policy="omit")\n        self.cols_ = np.where(np.nan_to_num(np.abs(np.asarray(sk))) > self.thr)[0]\n        return self\n\n    def transform(self, X):\n        X = np.array(X, float, copy=True)\n        if len(self.cols_):\n            X[:, self.cols_] = np.sign(X[:, self.cols_]) * np.log1p(np.abs(X[:, self.cols_]))\n        return X\n\n\ndef _split_cols(X):\n    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]\n    return num, [c for c in X.columns if c not in num]\n\n\ndef make_prep(X, family):\n    """family: \'hgb\' (native NaN), \'tree\' (impute) or \'linear\' (impute + skew fix + scale)."""\n    num, cat = _split_cols(X)\n    has_missing = bool(X[num].isna().any().any()) if num else False\n    parts = []\n    if num:\n        if family == "hgb":\n            parts.append(("num", "passthrough", num))\n        elif family == "tree":\n            parts.append(("num", SimpleImputer(strategy="median", add_indicator=has_missing,\n                                               keep_empty_features=True), num))\n        else:\n            parts.append(("num", Pipeline([\n                ("imp", SimpleImputer(strategy="median", add_indicator=has_missing, keep_empty_features=True)),\n                ("skew", SkewFix()), ("sc", StandardScaler())]), num))\n    if cat:\n        parts.append(("cat", Pipeline([\n            ("imp", SimpleImputer(strategy="constant", fill_value=MISSING_TOKEN)),\n            ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, max_categories=20,\n                                  sparse_output=False, drop="if_binary"))]), cat))\n    return ColumnTransformer(parts, remainder="drop", sparse_threshold=0.0, verbose_feature_names_out=False)\n\n\ndef est_dim(X):\n    num, cat = _split_cols(X)\n    return len(num) + sum(min(X[c].nunique(), 20) for c in cat)\n\n\n# =============================================================================== supervised candidates\ndef build_candidates(task, X, y, n_classes=None):\n    n = len(X)\n    dim = est_dim(X)\n    es = n >= 600\n    cands = OrderedDict()\n    P = lambda fam, m: Pipeline([("prep", make_prep(X, fam)), ("m", m)])\n    if task == "classification":\n        cands["hgb"] = P("hgb", HistGradientBoostingClassifier(\n            learning_rate=0.06, max_iter=300, l2_regularization=1.0, early_stopping=es,\n            n_iter_no_change=20, random_state=SEED))\n        cands["logreg"] = P("linear", LogisticRegression(C=1.0, max_iter=500))\n        cands["hgb_small"] = P("hgb", HistGradientBoostingClassifier(\n            learning_rate=0.1, max_iter=200, max_leaf_nodes=8, min_samples_leaf=10, l2_regularization=2.0,\n            early_stopping=es, n_iter_no_change=15, random_state=SEED))\n        cands["rf"] = P("tree", RandomForestClassifier(n_estimators=120, min_samples_leaf=1, n_jobs=1,\n                                                       random_state=SEED))\n        cands["et"] = P("tree", ExtraTreesClassifier(n_estimators=120, min_samples_leaf=1, n_jobs=1,\n                                                     random_state=SEED))\n        if n <= 30000:\n            cands["mlp"] = P("linear", MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True,\n                                                     max_iter=200, random_state=SEED))\n        if dim <= 40 and n <= 30000:\n            cands["knn"] = P("linear", KNeighborsClassifier(n_neighbors=15, weights="distance"))\n        if dim <= 22 and n <= 8000:\n            cands["logreg_poly"] = Pipeline([("prep", make_prep(X, "linear")),\n                                             ("poly", PolynomialFeatures(2, include_bias=False)),\n                                             ("sc", StandardScaler()),\n                                             ("m", LogisticRegression(C=0.2, max_iter=500))])\n    else:\n        cands["hgb"] = P("hgb", HistGradientBoostingRegressor(\n            learning_rate=0.06, max_iter=400, l2_regularization=1.0, early_stopping=es,\n            n_iter_no_change=20, random_state=SEED))\n        cands["ridge"] = P("linear", RidgeCV(alphas=np.logspace(-3, 3, 13)))\n        cands["hgb_small"] = P("hgb", HistGradientBoostingRegressor(\n            learning_rate=0.1, max_iter=250, max_leaf_nodes=8, min_samples_leaf=10, l2_regularization=2.0,\n            early_stopping=es, n_iter_no_change=15, random_state=SEED))\n        if np.nanmin(y) >= 0 and abs(float(pd.Series(y).skew())) > 1.0:\n            cands["hgb_log"] = TransformedTargetRegressor(\n                regressor=cands["hgb"], func=np.log1p, inverse_func=np.expm1, check_inverse=False)\n        cands["rf"] = P("tree", RandomForestRegressor(n_estimators=120, min_samples_leaf=1, n_jobs=1,\n                                                      random_state=SEED))\n        cands["et"] = P("tree", ExtraTreesRegressor(n_estimators=120, min_samples_leaf=1, n_jobs=1,\n                                                    random_state=SEED))\n        if n <= 30000:\n            cands["mlp"] = TransformedTargetRegressor(\n                regressor=P("linear", MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True,\n                                                   max_iter=300, random_state=SEED)),\n                transformer=StandardScaler())\n        if dim <= 40 and n <= 30000:\n            cands["knn"] = P("linear", KNeighborsRegressor(n_neighbors=10, weights="distance"))\n        if dim <= 30 and n <= 8000:\n            cands["ridge_poly"] = Pipeline([("prep", make_prep(X, "linear")),\n                                            ("poly", PolynomialFeatures(2, include_bias=False)),\n                                            ("sc", StandardScaler()),\n                                            ("m", RidgeCV(alphas=np.logspace(-1, 4, 11)))])\n    return cands\n\n\ndef _logloss(P, y):\n    P = np.clip(P, 1e-15, 1.0)\n    P = P / P.sum(1, keepdims=True)\n    return float(-np.mean(np.log(P[np.arange(len(y)), y])))\n\n\ndef _rmse(p, y):\n    return float(np.sqrt(np.mean((p - y) ** 2)))\n\n\ndef _fold_iter(task, y, k):\n    if task == "classification":\n        mc = np.bincount(y).min()\n        if mc >= 2:\n            return list(StratifiedKFold(int(min(k, mc)), shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))\n    return list(KFold(k, shuffle=True, random_state=SEED).split(np.zeros(len(y))))\n\n\ndef _classes_of(model):\n    return model.classes_ if hasattr(model, "classes_") else model[-1].classes_\n\n\ndef _predict_clf(model, Xv, n_classes):\n    p = model.predict_proba(Xv)\n    full = np.zeros((len(Xv), n_classes))\n    full[:, np.asarray(_classes_of(model), int)] = p\n    return full\n\n\ndef greedy_blend(oofs, loss_fn, steps=20):\n    chosen, cur, best_loss = [], None, np.inf\n    for _ in range(steps):\n        best = None\n        for nm, o in oofs.items():\n            cand = o if cur is None else (cur * len(chosen) + o) / (len(chosen) + 1)\n            l = loss_fn(cand)\n            if best is None or l < best[0]:\n                best = (l, nm, cand)\n        if chosen and best[0] >= best_loss - 1e-9:\n            break\n        chosen.append(best[1])\n        cur, best_loss = best[2], best[0]\n    cnt = Counter(chosen)\n    tot = sum(cnt.values())\n    return {k: v / tot for k, v in cnt.items()}, best_loss\n\n\ndef _decision_score(y, pred):\n    return float(np.mean([accuracy_score(y, pred), balanced_accuracy_score(y, pred),\n                          f1_score(y, pred, average="macro")]))\n\n\ndef tune_class_weights(P, y):\n    """Prior-shift weights for imbalanced data, tuned on out-of-fold probabilities."""\n    K = P.shape[1]\n    w = np.ones(K)\n    base = _decision_score(y, (P * w).argmax(1))\n    best = base\n    grid = [0.25, 0.4, 0.6, 0.8, 1.25, 1.6, 2.5, 4.0]\n    for _ in range(2):\n        for k in range(K):\n            for g in grid:\n                w2 = w.copy()\n                w2[k] *= g\n                sc = _decision_score(y, (P * w2).argmax(1))\n                if sc > best + 1e-6:\n                    best, w = sc, w2\n    if best - base < 0.004:  # not a real gain -> keep plain argmax\n        return np.ones(K), base, base\n    return w / w.max(), base, best\n\n\ndef run_supervised(Xtr, ytr_raw, Xte, task, budget, info):\n    n_full = len(Xtr)\n    if task == "classification":\n        le = LabelEncoder()\n        y = le.fit_transform(ytr_raw)\n        K = len(le.classes_)\n    else:\n        y = np.asarray(ytr_raw, float)\n        le, K = None, None\n\n    if n_full < 8 or (task == "classification" and K < 2) or Xtr.shape[1] == 0:\n        return _prior_prediction(task, y, le, K, len(Xte), info)\n\n    # selection subsample\n    sel_n = 6000\n    if n_full > sel_n:\n        rng = np.random.RandomState(SEED)\n        if task == "classification":\n            from sklearn.model_selection import train_test_split\n            idx, _ = train_test_split(np.arange(n_full), train_size=sel_n, stratify=y, random_state=SEED) \\\n                if np.bincount(y).min() >= 2 else (rng.choice(n_full, sel_n, replace=False), None)\n        else:\n            idx = rng.choice(n_full, sel_n, replace=False)\n        idx = np.sort(idx)\n    else:\n        idx = np.arange(n_full)\n    Xs, ys = Xtr.iloc[idx].reset_index(drop=True), y[idx]\n    n_sel = len(idx)\n    k = 5 if n_sel <= 3000 else 3\n    folds = _fold_iter(task, ys, k)\n\n    cands = build_candidates(task, Xs, ys, K)\n    loss_fn = (lambda P: _logloss(P, ys)) if task == "classification" else (lambda p: _rmse(p, ys))\n    oofs, cv_time, errors = {}, {}, {}\n    for name, est in cands.items():\n        if oofs and not budget.select_open():\n            break\n        oof = np.zeros((n_sel, K)) if task == "classification" else np.zeros(n_sel)\n        t0 = time.time()\n        try:\n            aborted = False\n            for tr, va in folds:\n                m = clone(est).fit(Xs.iloc[tr], ys[tr])\n                if task == "classification":\n                    oof[va] = _predict_clf(m, Xs.iloc[va], K)\n                else:\n                    oof[va] = np.asarray(m.predict(Xs.iloc[va]), float)\n                if oofs and budget.select_hard_stop():\n                    aborted = True\n                    break\n            if aborted or not np.all(np.isfinite(oof)):\n                continue\n            oofs[name] = oof\n            cv_time[name] = time.time() - t0\n        except Exception as e:\n            errors[name] = str(e)[:160]\n    if errors:\n        info["model_errors"] = errors\n    if not oofs:\n        return _prior_prediction(task, y, le, K, len(Xte), info)\n\n    scores = {nm: loss_fn(o) for nm, o in oofs.items()}\n    weights, blend_loss = greedy_blend(oofs, loss_fn)\n    info["cv_metric"] = "log_loss" if task == "classification" else "rmse"\n    info["cv_scores"] = {k_: round(v, 5) for k_, v in sorted(scores.items(), key=lambda kv: kv[1])}\n    info["blend_weights"] = {k_: round(v, 3) for k_, v in weights.items()}\n    info["blend_cv_score"] = round(blend_loss, 5)\n    info["n_rows_used_for_selection"] = int(n_sel)\n    info["cv_seconds"] = {k_: round(v, 2) for k_, v in cv_time.items()}\n\n    blend_oof = sum(weights[nm] * oofs[nm] for nm in weights)\n    if task == "classification":\n        info["cv_accuracy"] = round(float(accuracy_score(ys, blend_oof.argmax(1))), 4)\n        info["cv_balanced_accuracy"] = round(float(balanced_accuracy_score(ys, blend_oof.argmax(1))), 4)\n        cw = np.ones(K)\n        share = np.bincount(ys, minlength=K) / n_sel\n        if share.max() / max(share.min(), 1e-9) > 2.5 and np.bincount(ys).min() >= 5:\n            cw, base, best = tune_class_weights(blend_oof, ys)\n            info["class_weight_tuning"] = {"weights": [round(float(x), 3) for x in cw],\n                                           "score_before": round(base, 4), "score_after": round(best, 4)}\n    else:\n        info["cv_r2"] = round(float(r2_score(ys, blend_oof)), 4)\n    # linear vs non-linear signal (profile hint)\n    lin = [scores[m] for m in scores if m in ("logreg", "ridge")]\n    non = [scores[m] for m in scores if m in ("hgb", "hgb_small", "rf", "et", "mlp", "knn", "hgb_log")]\n    if lin and non:\n        info["nonlinear_signal"] = bool(min(non) < (0.95 if task == "classification" else 0.9) * min(lin))\n\n    # final fits on all rows, respecting the clock\n    order = sorted(weights, key=lambda m: -weights[m])\n    preds, used = {}, {}\n    scale = n_full / n_sel\n    for nm in order:\n        est_t = cv_time[nm] / len(folds) * scale * 1.3\n        if preds and est_t > budget.left() - 4:\n            continue\n        try:\n            m = clone(cands[nm]).fit(Xtr, y)\n            preds[nm] = _predict_clf(m, Xte, K) if task == "classification" else np.asarray(m.predict(Xte), float)\n            used[nm] = weights[nm]\n        except Exception as e:\n            info.setdefault("model_errors", {})[nm + "_final"] = str(e)[:160]\n    if not preds:  # last resort: fastest scored model\n        nm = min(cv_time, key=cv_time.get)\n        m = clone(cands[nm]).fit(Xtr, y)\n        preds[nm] = _predict_clf(m, Xte, K) if task == "classification" else np.asarray(m.predict(Xte), float)\n        used[nm] = 1.0\n    tot = sum(used.values())\n    final = sum(used[nm] / tot * preds[nm] for nm in preds)\n    info["chosen_models"] = {k_: round(v / tot, 3) for k_, v in used.items()}\n    if task == "classification":\n        w = np.array(info.get("class_weight_tuning", {}).get("weights", np.ones(K)))\n        out = le.inverse_transform((final * w).argmax(1))\n        return out, final, le.classes_\n    if np.nanmin(y) >= 0:\n        final = np.clip(final, 0, None)\n    return final, None, None\n\n\ndef _prior_prediction(task, y, le, K, n_out, info):\n    info["note"] = "fallback to prior (too little data or no usable features)"\n    info["chosen_models"] = {"prior": 1.0}\n    if task == "classification":\n        c = np.bincount(y, minlength=K) / len(y)\n        proba = np.tile(c, (n_out, 1))\n        return np.repeat(le.inverse_transform([int(np.argmax(c))]), n_out), proba, le.classes_\n    return np.full(n_out, float(np.mean(y)) if len(y) else 0.0), None, None\n\n\n# =============================================================================== clustering\ndef _structure_gain(Z, k):\n    """BIC gain of a k-component 1-D mixture over one Gaussian; positive => feature has cluster structure."""\n    gains = []\n    for j in range(Z.shape[1]):\n        v = Z[:, [j]]\n        try:\n            gains.append(GaussianMixture(1, random_state=SEED).fit(v).bic(v) -\n                         GaussianMixture(min(k, 4), random_state=SEED, reg_covar=1e-4).fit(v).bic(v))\n        except Exception:\n            gains.append(-1e9)\n    return np.array(gains)\n\n\ndef cluster_spaces(X, k):\n    num, cat = _split_cols(X)\n    spaces, meta = OrderedDict(), {}\n\n    def enc(cols, scale_fam="linear"):\n        sub = X[cols]\n        P = make_prep(sub, scale_fam)\n        Z = np.nan_to_num(P.fit_transform(sub))\n        return Z[:, Z.std(0) > 1e-8]\n\n    if num:\n        sub = X[num]\n        if sub.isna().any().any() and len(num) <= 60 and len(sub) <= 20000 and len(num) > 1:\n            imp = IterativeImputer(max_iter=8, random_state=SEED, skip_complete=True)\n        else:\n            imp = SimpleImputer(strategy="median", keep_empty_features=True)\n        Zn = np.nan_to_num(Pipeline([("imp", imp), ("skew", SkewFix()), ("sc", StandardScaler())]).fit_transform(sub))\n        Zn = Zn[:, Zn.std(0) > 1e-8]\n        names = [c for c, sd in zip(num, np.nan_to_num(Pipeline([("imp", SimpleImputer(strategy="median", keep_empty_features=True))]).fit_transform(sub)).std(0)) if sd > 1e-8]\n        if Zn.shape[1]:\n            spaces["num"] = Zn\n            if Zn.shape[1] > 3:\n                g = _structure_gain(Zn, k)\n                keep = np.argsort(-g)[:max(2, int(np.ceil(Zn.shape[1] / 2)))]\n                keep = [j for j in keep if g[j] > 0] or list(range(Zn.shape[1]))\n                spaces["num_sel"] = Zn[:, sorted(keep)]\n                meta["unstructured_features"] = [names[j] for j in range(min(len(names), len(g))) if g[j] <= 0]\n    if cat:\n        Za = enc(list(X.columns))\n        if Za.shape[1]:\n            spaces["all"] = Za\n            if Za.shape[1] > 6:\n                spaces["all_pca"] = PCA(n_components=0.9, random_state=SEED).fit_transform(Za)\n    return spaces, meta\n\n\ndef ensure_k(labels, k):\n    """Split the largest groups deterministically when identical points collapse clusters."""\n    labels = LabelEncoder().fit_transform(labels)\n    if not 1 <= k <= len(labels):\n        raise ValueError("n_clusters must be between 1 and the number of rows")\n    while len(np.unique(labels)) < k:\n        sizes = np.bincount(labels)\n        members = np.flatnonzero(labels == sizes.argmax())\n        labels[members[-1]] = len(sizes)\n    return labels\n\n\ndef run_clustering(X, k, budget, info):\n    n = len(X)\n    if not 1 <= k <= n:\n        raise ValueError("n_clusters must be between 1 and the number of rows")\n    if k == 1 or k == n:\n        info["note"] = "No varying features; deterministic partition has no learned structure"\n        return np.arange(n) % k if k == 1 else np.arange(n)\n    spaces, meta = cluster_spaces(X, k)\n    info.update(meta)\n    if not spaces:\n        info["note"] = "No varying features; deterministic partition has no learned structure"\n        return np.arange(n) % k\n    rng = np.random.RandomState(SEED)\n    sil_idx = rng.choice(n, min(n, 2500), replace=False)\n    cands = []  # (name, labels, silhouette, purity)\n    for sname, Z in spaces.items():\n        d = Z.shape[1]\n        nn_idx = NearestNeighbors(n_neighbors=min(11, max(2, len(sil_idx) - 1))).fit(Z[sil_idx]).kneighbors(\n            Z[sil_idx], return_distance=False)[:, 1:]\n        algos = OrderedDict()\n        algos["kmeans"] = lambda: KMeans(k, n_init=10, random_state=SEED)\n        for cov in ("spherical", "diag", "full"):\n            if cov == "full" and d > 15:\n                continue\n            algos[f"gmm_{cov}"] = (lambda c=cov: GaussianMixture(k, covariance_type=c, n_init=2, random_state=SEED,\n                                                                 reg_covar=1e-4))\n        if n <= 5000:\n            algos["ward"] = lambda: AgglomerativeClustering(k, linkage="ward")\n        if n <= 2000 and sname in ("num", "num_sel", "all") and d >= 2:\n            algos["spectral"] = lambda: SpectralClustering(\n                k, affinity="nearest_neighbors", n_neighbors=min(15, max(5, n // 50)), random_state=SEED)\n        for aname, mk in algos.items():\n            if cands and budget.elapsed() > 0.6 * budget.total:\n                break\n            try:\n                lab = mk().fit_predict(Z)\n                if len(np.unique(lab)) < 2:\n                    continue\n                s = float(silhouette_score(Z[sil_idx], lab[sil_idx])) if len(np.unique(lab[sil_idx])) > 1 else -1\n                ls = np.asarray(lab)[sil_idx]\n                purity = float(np.mean(ls[nn_idx] == ls[:, None]))\n                cands.append((f"{aname}/{sname}", np.asarray(lab), s, purity))\n            except Exception as e:\n                info.setdefault("model_errors", {})[f"{aname}/{sname}"] = str(e)[:160]\n    if not cands:\n        return ensure_k(KMeans(k, n_init=5, random_state=SEED).fit_predict(next(iter(spaces.values()))), k)\n    # consensus: how well does each labelling agree with all the others\n    m = len(cands)\n    agree = np.zeros(m)\n    for i in range(m):\n        if m > 1:\n            agree[i] = np.mean([adjusted_rand_score(cands[i][1], cands[j][1]) for j in range(m) if j != i])\n    scored = []\n    for i, (name, lab, s, pur) in enumerate(cands):\n        sizes = np.bincount(lab)\n        penalty = 0.0\n        if len(np.unique(lab)) != k:\n            penalty += 0.3\n        if sizes[sizes > 0].min() < 0.01 * n:\n            penalty += 0.15\n        scored.append((0.4 * s + 0.3 * agree[i] + 0.3 * pur - penalty, name, lab, s, agree[i], pur))\n    scored.sort(key=lambda t: -t[0])\n    best = scored[0]\n    info["cluster_candidates"] = {nm: {"score": round(sc, 4), "silhouette": round(s, 4), "consensus_ari": round(a, 4),\n                                       "neighbour_purity": round(p, 4)} for sc, nm, _, s, a, p in scored[:12]}\n    info["chosen_models"] = {best[1]: 1.0}\n    info["silhouette"] = round(best[3], 4)\n    return ensure_k(best[2], k)\n\n\n# =============================================================================== profiling\ndef _f(x):\n    try:\n        x = float(x)\n    except Exception:\n        return None\n    return None if np.isnan(x) or np.isinf(x) else round(x, 6)\n\n\ndef build_profile(raw_X, cleaner, y, task, k, target, info, Xclean, relevance=True):\n    basic = cleaner.basic_\n    n = len(raw_X)\n    num_cols = [c for c in basic.columns if cleaner.is_num_[c]]\n    cat_cols = [c for c in basic.columns if not cleaner.is_num_[c]]\n    raw_kind = cleaner.kinds_\n    prof = OrderedDict()\n    prof["task"] = task\n    prof["n_rows"] = int(n)\n    prof["n_columns"] = int(raw_X.shape[1])\n    if target is not None:\n        prof["target_column"] = target\n    if k:\n        prof["n_clusters_requested"] = int(k)\n    prof["column_types"] = {c: raw_kind[c] for c in raw_X.columns}\n    prof["numeric_columns"] = [c for c in raw_X.columns if raw_kind[c] in ("numeric", "numeric_text")]\n    prof["categorical_columns"] = [c for c in raw_X.columns if raw_kind[c] == "categorical"]\n    prof["boolean_columns"] = [c for c in raw_X.columns if raw_kind[c] == "bool"]\n    prof["datetime_columns"] = [c for c in raw_X.columns if raw_kind[c] == "datetime"]\n    prof["numeric_stored_as_text"] = [c for c in raw_X.columns if raw_kind[c] == "numeric_text"]\n    # One normalized mask supplies every missing-value statistic, including\n    # markers in datetime columns and nullable boolean values.\n    missing_mask = raw_X.isna().copy()\n    for c in raw_X.columns:\n        missing_mask[c] |= raw_X[c].astype("string").str.strip().isin(["", "?"])\n    miss = {c: int(missing_mask[c].sum()) for c in raw_X.columns}\n    prof["missing_counts"] = {c: v for c, v in miss.items() if v > 0}\n    prof["missing_fraction"] = {c: round(v / n, 4) for c, v in miss.items() if v > 0}\n    prof["columns_with_missing"] = list(prof["missing_counts"])\n    prof["total_missing_cells"] = int(sum(miss.values()))\n    prof["rows_with_any_missing"] = int(missing_mask.any(axis=1).sum())\n    prof["constant_columns"] = [c for c, r in cleaner.dropped_.items() if r == "constant"]\n    prof["duplicate_columns"] = cleaner.dup_pairs_\n    prof["perfectly_correlated_columns"] = cleaner.near_pairs_\n    prof["id_like_columns"] = [c for c, r in cleaner.dropped_.items() if r == "id_like"]\n    prof["n_duplicate_rows"] = int(raw_X.duplicated().sum())\n    prof["dropped_columns"] = dict(cleaner.dropped_)\n    prof["high_cardinality_columns"] = [c for c in cat_cols if basic[c].nunique() > 20]\n    prof["cardinality"] = {c: int(basic[c].nunique()) for c in cat_cols}\n    if num_cols:\n        B = basic[num_cols]\n        sd = B.std()\n        pos = sd[sd > 1e-12]\n        prof["numeric_summary"] = {c: {"mean": _f(B[c].mean()), "std": _f(sd[c]), "min": _f(B[c].min()),\n                                       "max": _f(B[c].max()), "skew": _f(B[c].skew())} for c in num_cols}\n        prof["scale_ratio_max_over_min_std"] = _f(pos.max() / pos.min()) if len(pos) > 1 else 1.0\n        prof["differently_scaled_features"] = bool(len(pos) > 1 and pos.max() / pos.min() > 20)\n        prof["skewed_columns"] = [c for c in num_cols if abs(_f(B[c].skew()) or 0) > 1.0]\n        q1, q3 = B.quantile(0.25), B.quantile(0.75)\n        iqr = q3 - q1\n        out_ct = ((B.lt(q1 - 3 * iqr)) | (B.gt(q3 + 3 * iqr))).sum()\n        prof["outlier_counts_3iqr"] = {c: int(v) for c, v in out_ct.items() if v > 0 and iqr[c] > 0}\n    prof["n_features_after_cleaning"] = int(Xclean.shape[1])\n    if y is not None:\n        if task == "classification":\n            vc = pd.Series(y).value_counts(normalize=True)\n            cnt = pd.Series(y).value_counts()\n            prof["n_classes"] = int(len(vc))\n            prof["class_counts"] = {str(a): int(b) for a, b in cnt.items()}\n            prof["class_distribution"] = {str(a): round(float(b), 4) for a, b in vc.items()}\n            prof["imbalance_ratio"] = _f(vc.max() / vc.min())\n            prof["minority_class"] = str(vc.idxmin())\n            prof["is_imbalanced"] = bool(vc.max() / vc.min() > 3)\n        else:\n            ys = pd.Series(y, dtype=float)\n            prof["target_summary"] = {s: _f(getattr(ys, s)()) for s in ("mean", "std", "min", "max", "skew")}\n            prof["target_skewed"] = bool(abs(_f(ys.skew()) or 0) > 1.0)\n        prof["likely_irrelevant_features"] = _irrelevant(Xclean, y, task) if relevance else []\n        prof["irrelevance_analysis_completed"] = bool(relevance)\n    prof["model_report"] = info\n    return prof\n\n\ndef _irrelevant(X, y, task, budget_rows=1500, max_cols=150):\n    """Features whose mutual information with the target is no better than with a shuffled target."""\n    try:\n        if X.shape[1] == 0 or len(y) < 30:\n            return []\n        cols = list(X.columns)[:max_cols]\n        rng = np.random.RandomState(SEED)\n        idx = rng.choice(len(X), min(len(X), budget_rows), replace=False)\n        S = X.iloc[idx][cols]\n        M = np.zeros((len(S), len(cols)))\n        disc = np.zeros(len(cols), bool)\n        for j, c in enumerate(cols):\n            if pd.api.types.is_numeric_dtype(S[c]):\n                v = S[c].to_numpy(float)\n                M[:, j] = np.where(np.isnan(v), np.nanmedian(v) if np.isfinite(v).any() else 0, v)\n            else:\n                M[:, j] = pd.factorize(S[c].fillna(MISSING_TOKEN))[0]\n                disc[j] = True\n        yy = np.asarray(y)[idx]\n        if task == "classification":\n            yy = pd.factorize(yy)[0]\n            f = lambda t: mutual_info_classif(M, t, discrete_features=disc, random_state=SEED, n_neighbors=3)\n        else:\n            yy = yy.astype(float)\n            f = lambda t: mutual_info_regression(M, t, discrete_features=disc, random_state=SEED, n_neighbors=3)\n        mi = f(yy)\n        null = f(rng.permutation(yy))\n        thr = np.percentile(null, 90) * 1.5 + 0.005\n        return [c for c, v in zip(cols, mi) if v <= thr]\n    except Exception:\n        return []\n\n\n# =============================================================================== I/O\ndef read_table(path):\n    ext = os.path.splitext(path)[1].lower()\n    if ext == ".parquet":\n        return pd.read_parquet(path)\n    if ext in (".json",):\n        return pd.read_json(path)\n    if ext in (".xlsx", ".xls"):\n        return pd.read_excel(path)\n    # Never let automatic inference choose letters from a one-column header\n    # as separators. Only common table delimiters are eligible.\n    delimiter = "\\t" if ext == ".tsv" else ","\n    if ext != ".tsv":\n        with open(path, "r", encoding="utf-8-sig", newline="") as stream:\n            sample = stream.read(65536)\n        try:\n            delimiter = csv.Sniffer().sniff(sample, delimiters=",;\\t|").delimiter\n        except csv.Error:\n            pass  # A genuine single-column CSV has no delimiter to detect.\n    return pd.read_csv(path, sep=delimiter)\n\n\ndef write_predictions(out_dir, ids, labels, proba, classes, task, id_name="row_id"):\n    os.makedirs(out_dir, exist_ok=True)\n    df = pd.DataFrame({id_name: ids, "cluster" if task == "clustering" else "prediction": labels})\n    if proba is not None:\n        for i, c in enumerate(classes):\n            df[f"proba_{c}"] = np.round(proba[:, i], 6)\n    df.to_csv(os.path.join(out_dir, "predictions.csv"), index=False)\n\n\n# =============================================================================== orchestration\ndef infer_task(y):\n    if not pd.api.types.is_numeric_dtype(y) or pd.api.types.is_bool_dtype(y):\n        return "classification"\n    nun = y.nunique(dropna=True)\n    integral = np.allclose(y.dropna() % 1, 0)\n    if integral and nun <= max(15, 0.02 * len(y)):\n        return "classification"\n    return "regression"\n\n\ndef solve(train, test=None, target=None, task=None, n_clusters=None, id_col=None, time_budget=40.0, checkpoint=None):\n    budget = Budget(time_budget)\n    if len(train) == 0:\n        raise ValueError("training data is empty")\n    info = {}\n    if task is None:\n        task = "clustering" if (n_clusters and target is None) else None\n    y, target_name = None, None\n    if task != "clustering":\n        target_name = target or train.columns[-1]\n        y = train[target_name]\n        keep = y.notna().to_numpy()\n        train = train.loc[keep].reset_index(drop=True)\n        y = y.loc[keep].reset_index(drop=True)\n        if len(y) == 0:\n            raise ValueError("training target contains no observed labels")\n        task = task or infer_task(y)\n        if task == "classification" and pd.api.types.is_float_dtype(y) and np.allclose(y % 1, 0):\n            y = y.astype(int)\n        X = train.drop(columns=[target_name])\n    else:\n        X = train.copy()\n    out_ids, out_id_name = None, "row_id"\n    if id_col and id_col in X.columns:\n        if test is None:\n            out_ids, out_id_name = X[id_col].to_numpy(), id_col\n        X = X.drop(columns=[id_col])\n    if test is not None:\n        if id_col and id_col in test.columns:\n            out_ids, out_id_name = test[id_col].to_numpy(), id_col\n            test = test.drop(columns=[id_col])\n        if target_name and target_name in test.columns:\n            test = test.drop(columns=[target_name])\n\n    cleaner = FeatureCleaner().fit(X)\n    Xc = cleaner.transform(X)\n\n    if checkpoint is not None:\n        Xt0 = Xc if test is None else cleaner.transform(test)\n        fallback_info = {"chosen_models": {"prior": 1.0}, "checkpoint": True}\n        probs0 = classes0 = None\n        if task == "clustering":\n            k0 = int(n_clusters or 0)\n            total_rows = len(Xc) + (len(Xt0) if test is not None else 0)\n            if not 1 <= k0 <= total_rows:\n                raise ValueError("invalid n_clusters")\n            labels0 = np.arange(total_rows) % k0\n            if test is not None:\n                labels0 = labels0[len(Xc):]\n            fallback_info["note"] = "Emergency partition; no learned cluster structure"\n        elif task == "classification":\n            le0 = LabelEncoder().fit(y)\n            labels0, probs0, classes0 = _prior_prediction(task, le0.transform(y), le0, len(le0.classes_), len(Xt0), fallback_info)\n        else:\n            labels0 = np.full(len(Xt0), np.mean(np.asarray(y, float)))\n        ids0 = out_ids if out_ids is not None else np.arange(len(labels0))\n        p0 = build_profile(X, cleaner, y, task, n_clusters, target_name, fallback_info, Xc, relevance=False)\n        def emit0():\n            fallback_info["runtime_seconds"] = round(budget.elapsed(), 2)\n            checkpoint(dict(task=task, ids=ids0, id_name=out_id_name, labels=labels0,\n                            proba=probs0, classes=classes0, profile=p0))\n        emit0()\n        if task != "clustering" and Xc.shape[1] and budget.select_open():\n            try:\n                from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor\n                model0 = DecisionTreeClassifier(max_depth=8, random_state=SEED) if task == "classification" else DecisionTreeRegressor(max_depth=8, random_state=SEED)\n                fast0 = Pipeline([("prep", make_prep(Xc, "tree")), ("model", model0)])\n                fast0.fit(Xc, y)\n                labels0 = fast0.predict(Xt0)\n                if task == "classification":\n                    probs0, classes0 = fast0.predict_proba(Xt0), fast0.classes_\n                fallback_info.update(chosen_models={"emergency_tree": 1.0}, note="Quick learned checkpoint before model search")\n                emit0()\n            except Exception:\n                pass\n\n    if task == "clustering":\n        if not n_clusters:\n            raise SystemExit("clustering needs --n-clusters")\n        Xl = Xc if test is None else pd.concat([Xc, cleaner.transform(test)], ignore_index=True)\n        labels = run_clustering(Xl, int(n_clusters), budget, info)\n        if test is not None:\n            labels = labels[len(Xc):]\n        ids = out_ids if out_ids is not None else np.arange(len(labels))\n        proba = classes = None\n    else:\n        Xt = Xc if test is None else cleaner.transform(test)\n        labels, proba, classes = run_supervised(Xc, y, Xt, task, budget, info)\n        ids = out_ids if out_ids is not None else np.arange(len(Xt))\n\n    info["runtime_seconds"] = round(budget.elapsed(), 2)\n    profile = build_profile(X, cleaner, y, task, n_clusters, target_name, info, Xc)\n    info["runtime_seconds"] = round(budget.elapsed(), 2)\n    cleaner.basic_ = None\n    return dict(task=task, ids=ids, id_name=out_id_name, labels=labels, proba=proba, classes=classes,\n                profile=profile)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--train", required=True, help="training table (or the table to cluster)")\n    ap.add_argument("--test", default=None, help="table to predict (default: the training table)")\n    ap.add_argument("--target", default=None, help="target column (default: last column)")\n    ap.add_argument("--task", choices=["classification", "regression", "clustering"], default=None)\n    ap.add_argument("--n-clusters", type=int, default=None)\n    ap.add_argument("--id-col", default=None)\n    ap.add_argument("--out-dir", default="out")\n    ap.add_argument("--time-budget", type=float, default=40.0, help="soft budget in seconds (hard limit 60)")\n    a = ap.parse_args(argv)\n\n    train = read_table(a.train)\n    test = read_table(a.test) if a.test else None\n    checkpoint_count = 0\n    def save(r):\n        nonlocal checkpoint_count\n        checkpoint_count += 1\n        folder = os.path.join(a.out_dir, "checkpoint_" + str(checkpoint_count))\n        write_predictions(folder, r["ids"], r["labels"], r["proba"], r["classes"], r["task"], r["id_name"])\n        with open(os.path.join(folder, "profile.json"), "w") as f:\n            json.dump(r["profile"], f, indent=2, default=str)\n        marker = os.path.join(a.out_dir, "checkpoint.json")\n        with open(marker + ".tmp", "w") as f:\n            json.dump({"directory": os.path.basename(folder)}, f)\n        os.replace(marker + ".tmp", marker)\n    r = solve(train, test, a.target, a.task, a.n_clusters, a.id_col, a.time_budget, checkpoint=save)\n    write_predictions(a.out_dir, r["ids"], r["labels"], r["proba"], r["classes"], r["task"], r["id_name"])\n    with open(os.path.join(a.out_dir, "profile.json"), "w") as f:\n        json.dump(r["profile"], f, indent=2, default=str)\n    rep = r["profile"]["model_report"]\n    print(json.dumps({"task": r["task"], "runtime_s": rep["runtime_seconds"], "models": rep.get("chosen_models"),\n                      "cv": rep.get("blend_cv_score", rep.get("silhouette"))}))\n\n\nif __name__ == "__main__":\n    main()\n', 'real_results.json': '[\n  {\n    "dataset": "Telco",\n    "wall_seconds": 8.502,\n    "models": {\n      "hgb_small": 0.667,\n      "logreg": 0.333\n    },\n    "timeout": false,\n    "accuracy": 0.7864849517319704,\n    "balanced_accuracy": 0.7507057776130319,\n    "macro_f1": 0.7383994917616716,\n    "roc_auc": 0.8498472607885514\n  },\n  {\n    "dataset": "Maintenance",\n    "wall_seconds": 6.5,\n    "models": {\n      "hgb_small": 0.75,\n      "hgb": 0.25\n    },\n    "timeout": false,\n    "accuracy": 0.9816,\n    "balanced_accuracy": 0.8939958592132505,\n    "macro_f1": 0.868852608578636,\n    "roc_auc": 0.9718182925344051\n  },\n  {\n    "dataset": "Mall",\n    "wall_seconds": 1.857,\n    "models": {\n      "gmm_full/num": 1.0\n    },\n    "timeout": false,\n    "clusters": 5,\n    "note": "Five clusters assumed for this example; no ground-truth labels supplied."\n  }\n]', 'README.md': '# Unknown Dataset Hackathon — submission\n\n**Team:** Georgio Sleiman, Andrew Sleiman, Carla Doumit, Dimitri Antypas\n\n**GitHub repository:** https://github.com/DimitriAntypas/unknown-dataset-hackathon\n\n**Google Colab notebook:** Included as `Unknown_Dataset_Hackathon.ipynb`; a shared Colab link is optional.\n\nThe GitHub repository above is the submission link. The included notebook can also run in Colab;\na shared Colab link is optional unless separately requested. The brief specifies three students per team;\nconfirm an exception before submitting the four names above.\n\n- `Unknown_Dataset_Hackathon.ipynb`: self-contained Google Colab notebook. Upload it to Colab and run the numbered cells.\n- `main.py` and `supervisor.py`: command-line entry point and timeout supervisor; keep both at the repository root.\n- `requirements.txt`: pinned dependencies.\n- `UPLOAD_INSTRUCTIONS.md`: exact upload and submission steps.\n- `VALIDATION.md`: test results and known limitations.\n\n\n# Unknown Dataset Hackathon – automatic ML for unfamiliar tables\n\n**Team:** Georgio Sleiman, Andrew Sleiman, Carla Doumit, Dimitri Antypas  |  **Entry point:** `python main.py`  |  **Python:** 3.12 (3.11+ required by the pinned versions)\n\nOne program that inspects an unfamiliar table, chooses its own preparation and model, and writes predictions plus a\nstructured data profile. It covers **classification, regression and clustering** with no per-dataset configuration.\n\n## Quick start\n```bash\npython -m venv .venv && source .venv/bin/activate      # Windows: .venv\\Scripts\\activate\npip install -r requirements.txt\npython main.py --train train.csv --test test.csv --target y --out-dir out                 # supervised\npython main.py --train data.csv --task clustering --n-clusters 4 --out-dir out            # clustering\n```\n| flag | meaning |\n|---|---|\n| `--train` | training table (or the table to cluster) |\n| `--test` | table to predict (defaults to the training table) |\n| `--target` | target column (default: last column) |\n| `--task` | `classification` / `regression` / `clustering` (inferred from the target if omitted) |\n| `--n-clusters` | required for clustering |\n| `--id-col` | column copied into the output under its original name |\n| `--time-budget` | search budget in seconds (default 40); CLI worker cutoff is min(budget + 8, 52) |\n\nOutputs: `out/predictions.csv` (`row_id`, or the supplied ID column name, `prediction` or `cluster`, plus `proba_<class>` columns) and\n`out/profile.json`.\n\n## How it works\n1. **Cleaning (`FeatureCleaner`)** – learns types on the training table and applies the same plan to any test table:\n   numbers stored as text (e.g. `" "` blanks), datetime strings (expanded to year/month/day/weekday/hour/epoch),\n   booleans, `?`/blank missing markers, infinities; drops constant, empty, exact-duplicate, perfectly-correlated\n   (scaled copies) and ID-like columns; frequency-encodes high-cardinality categoricals; tolerates missing columns at\n   test time.\n2. **Model-specific preprocessing** – native-NaN passthrough for gradient boosting, median imputation (+ missing\n   indicators) for forests, imputation + signed-log skew fix + scaling for linear / neural / kNN models; one-hot with\n   rare-level grouping and unseen-level handling.\n3. **Supervised search** – cross-validated out-of-fold predictions for up to 9 candidates (two gradient-boosting\n   variants, log-target boosting for skewed targets, random forest, extra trees, linear/logistic, polynomial ridge /\n   logistic, MLP, kNN). A **greedy ensemble selection** on the out-of-fold predictions (log-loss / RMSE) picks the blend\n   weights; chosen models are refit on all rows. Imbalanced classification then tunes class prior weights on the\n   out-of-fold probabilities against a hedge of accuracy, balanced accuracy and macro-F1.\n4. **Clustering** – several feature spaces (all numeric, a relevance-filtered subset, one-hot mix, PCA) with model-based\n   imputation × k-means, Gaussian mixtures (spherical/diag/full), Ward, spectral. The winner is chosen by silhouette,\n   consensus with the other labelings and nearest-neighbour purity, with penalties for empty/tiny clusters.\n5. **Runtime supervision** – the CLI starts a lightweight parent process before importing scientific libraries.\n   The worker saves a complete prior checkpoint and, for supervised data, a quick learned tree checkpoint before\n   the longer search. At `min(time_budget + 8, 52)` seconds the parent kills a stalled worker and publishes its latest\n   complete checkpoint. The 52-second cap leaves headroom below the contest\'s 60-second limit. This is a tested\n   safeguard, not a guarantee on arbitrary hardware or unbounded tables; no checkpoint may be available if initial\n   reading/cleaning itself exhausts the allowance. Direct Python `solve()` calls have a fresh soft budget each time;\n   use the CLI for process-level timeout protection. The CLI fixes numerical-library and forest concurrency to one thread.\n6. **Profile (`profile.json`)** – task, shape, column types (incl. text-numbers, datetimes), missing counts/fractions,\n   constant / duplicate / perfectly-correlated / ID-like columns, duplicate rows, cardinality, scale ratio and\n   `differently_scaled_features`, skewed columns, outlier counts, class distribution and `imbalance_ratio`, target\n   statistics, `likely_irrelevant_features` (mutual information vs a shuffled-target baseline), non-linear-signal flag,\n   plus the model report (CV scores, blend weights, timings).\n\n## Validation and reproducibility\n\nRun the complete checks in a fresh environment:\n```bash\npip install -r requirements.txt\npip install pytest\npython -m pytest -q\npython make_synthetic.py\npython evaluate.py\n```\nWithout pytest, run `python tests/test_pipeline.py`, `python tests/test_fixes.py`, and\n`python tests/test_review_fixes.py`.\nThe benchmark runner now returns failure if any dataset crashes or exceeds its 60-second subprocess limit.\nA successful benchmark run confirms execution; it does not enforce an official quality threshold.\n\nFor the three supplied public examples, extract their CSVs into one data directory and run:\n```bash\npython validate_real.py --data-dir /path/to/data --report real_results.json\n```\nThis uses a fixed stratified 75/25 holdout for classification. For predictive maintenance, the five failure-type\nindicators and Product ID are excluded to avoid an unrepresentative test using outcome-related fields.\nMall uses five clusters as a benchmark assumption; no ground-truth cluster labels are provided.\nSee `VALIDATION.md` and `real_results.json` for measured results and limitations.\nThese public examples and generated synthetic tables are not the nine official kickoff datasets.\n\n## Assumptions (reviewed at kickoff)\n- Input is CSV (comma, semicolon, tab and pipe delimiters are supported, as is JSON; Parquet/Excel need `pyarrow`/`openpyxl`, which are not pinned); the target is `--target` or the last column.\n- Output is `row_id, prediction` (or `cluster`). If the official contract differs, adapt the CLI, metadata parsing, `read_table`, `write_predictions`, and profile schema as necessary. The official checker has not been provided.\n- The dataset states its task; inferring it from the target is only a fallback.\n- CLI uses one thread, no network at run time, and fixed random seeds. Time-dependent model selection can still vary with machine speed and load.\n- The scoring metric for classification is not known in advance, so imbalanced problems are tuned for a blend of\n  accuracy, balanced accuracy and macro-F1.\n\n## Fixes in this revision\n- Sorted, unique integer measurements are retained unless their name identifies them as an ID.\n- Boolean features accept native values, boolean strings and missing markers consistently in test data.\n- Delimiter detection is restricted to comma, semicolon, tab and pipe, preserving single-column CSVs.\n- Missing counts, fractions, cell totals and row totals use one mask, including datetime missing markers.\n- Each `solve()` invocation starts a new monotonic budget.\n- CLI worker timeout recovery publishes a completed checkpoint, with `supervisor_timeout` recorded.\n- Clustering validates k and handles k=1, k=n, constant data, and collapsed duplicate-point clusters.\n  On data with no separating information, forced groups are deterministic partitions, not evidence of real structure.\n- Missing-value row counts include both blank and `?` markers.\n- ID values are retained even when no separate test table is supplied.\n- Empty training data and entirely missing targets produce clear errors.\n- Tests cover these cases and a deliberately stalled worker.\n\n## Known limitations\n- Clustering selection favours compact (convex) groups; strongly non-convex shapes (e.g. two moons) may be split\n  incorrectly.\n- Very large tables rely on subsampling for model selection (final fits use every row).\n\n## Tests\n`python -m pytest -q` (or the three test scripts above without pytest) covers string labels, missing test columns,\ntext-numbers, datetimes, constant/duplicate/ID columns, tiny and single-class data, high cardinality, clustering with\nmissing values, and the CLI end to end. The eight checks in `tests/test_review_fixes.py` specifically cover the\nfour corrected defects, including prediction quality on unseen sorted integer measurements, boolean test\ncolumns with missing markers, quoted single-column CSVs, supported delimiters and datetime missing counts.\n', 'make_synthetic.py': '"""Benchmark tables mimicking the hackathon\'s difficulties (scale, missing, constant/duplicate/ID/noise\ncolumns, imbalance, non-linearity, datetimes). Writes synth/*.csv and synth/manifest.json."""\nimport json, os\nimport numpy as np, pandas as pd\nfrom sklearn import datasets as D\n\nrng = np.random.RandomState(0)\nos.makedirs("synth", exist_ok=True)\nmanifest = []\n\n\ndef mess(df, noise=3, n_cat=2, miss=0.08, dup=True):\n    df = df.copy()\n    first = df.columns[0]\n    df[first] = df[first] * 1000                      # wild scale\n    df["const_col"] = 7\n    if dup:\n        df["dup_col"] = df[df.columns[1]] * 2.5 + 3   # scaled duplicate\n    df["row_id"] = np.arange(len(df))\n    for j in range(noise):\n        df[f"noise{j}"] = rng.randn(len(df))\n    for j in range(n_cat):\n        df[f"cat{j}"] = rng.choice(list("abcde"), len(df))\n    for c in list(df.columns[:3]) + ["cat0"]:\n        df.loc[rng.rand(len(df)) < miss, c] = np.nan\n    return df\n\n\ndef save_sup(name, task, X, y, test_frac=0.4, **kw):\n    df = mess(pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])]), **kw)\n    df["target"] = y\n    idx = rng.permutation(len(df)); cut = int(len(df) * (1 - test_frac))\n    tr, te = df.iloc[idx[:cut]], df.iloc[idx[cut:]]\n    tr.to_csv(f"synth/{name}_train.csv", index=False)\n    te.drop(columns="target").to_csv(f"synth/{name}_test.csv", index=False)\n    te[["target"]].to_csv(f"synth/{name}_truth.csv", index=False)\n    manifest.append(dict(name=name, task=task))\n\n\ndef save_clu(name, X, y, k, **kw):\n    df = mess(pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])]), **kw)\n    df.to_csv(f"synth/{name}.csv", index=False)\n    pd.DataFrame({"y": y}).to_csv(f"synth/{name}_truth.csv", index=False)\n    manifest.append(dict(name=name, task="clustering", k=k))\n\n\n# --- classification\nd = D.load_breast_cancer(); save_sup("bc", "classification", d.data[:, :12], d.target)\nd = D.load_wine(); save_sup("wine", "classification", d.data, d.target)\nX, y = D.make_classification(3000, 14, n_informative=5, weights=[0.9, 0.1], flip_y=0.02, random_state=1)\nsave_sup("imb", "classification", X, y)\nX, y = D.make_moons(1500, noise=0.25, random_state=1); X = np.hstack([X, rng.randn(1500, 4)])\nsave_sup("moons", "classification", X, y, noise=2)\n# --- regression\nd = D.load_diabetes(); save_sup("diab", "regression", d.data, d.target)\nX, y = D.make_friedman1(2500, 10, noise=1.0, random_state=1); save_sup("fried", "regression", X, y)\nX = rng.rand(3000, 4); dates = pd.Timestamp("2020-01-01") + pd.to_timedelta(rng.randint(0, 24 * 700, 3000), unit="h")\ny = 50 + 30 * np.sin(2 * np.pi * dates.hour / 24) + 15 * (dates.dayofweek >= 5) + 40 * X[:, 0] + rng.randn(3000) * 4\ndf = pd.DataFrame(X, columns=list("abcd")); df.insert(0, "datetime", dates.strftime("%Y-%m-%d %H:%M:%S")); df["count"] = np.exp(y / 40)\nidx = rng.permutation(3000); df.iloc[idx[:1800]].to_csv("synth/bike_train.csv", index=False)\ndf.iloc[idx[1800:]].drop(columns="count").to_csv("synth/bike_test.csv", index=False)\ndf.iloc[idx[1800:]][["count"]].rename(columns={"count": "target"}).to_csv("synth/bike_truth.csv", index=False)\nmanifest.append(dict(name="bike", task="regression", target="count"))\n# --- clustering\nd = D.load_iris(); save_clu("iris", d.data, d.target, 3, noise=2)\nX, y = D.make_blobs(900, 5, centers=4, cluster_std=1.3, random_state=0); X = np.hstack([X, rng.randn(900, 3)])\nsave_clu("blobs", X, y, 4)\nX, y = D.make_moons(800, noise=0.07, random_state=0); save_clu("moons_c", X, y, 2, noise=1, n_cat=0, miss=0.0)\nX, y = D.make_blobs(1200, 6, centers=5, cluster_std=[1, 2, 1.5, 0.7, 1.2], random_state=3); save_clu("blobs5", X, y, 5, noise=4)\njson.dump(manifest, open("synth/manifest.json", "w"))\n', 'tests/test_pipeline.py': '"""Robustness tests: python -m pytest -q"""\nimport json\nimport os\nimport subprocess\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\nimport main as M  # noqa: E402\n\nrng = np.random.RandomState(0)\n\n\ndef _clf_frame(n=300):\n    X = pd.DataFrame({"a": rng.randn(n), "b": rng.randn(n) * 1e4, "c": rng.choice(["x", "y", "z"], n)})\n    y = ((X["a"] + (X["c"] == "x") * 1.5 + rng.randn(n) * 0.3) > 0.5).astype(int)\n    return X.assign(target=np.where(y == 1, "yes", "no"))\n\n\ndef test_string_labels_and_missing_test_columns():\n    df = _clf_frame()\n    test = df.drop(columns=["target", "b"]).iloc[:50]       # column b absent in test\n    r = M.solve(df, test, target="target", time_budget=25)\n    assert set(r["labels"]) <= {"yes", "no"} and len(r["labels"]) == 50\n    assert r["profile"]["task"] == "classification"\n\n\ndef test_numeric_text_with_blanks_and_datetimes():\n    n = 400\n    df = pd.DataFrame({\n        "when": pd.date_range("2021-01-01", periods=n, freq="6h").strftime("%Y-%m-%d %H:%M"),\n        "charges": [" " if i % 25 == 0 else f"{rng.rand() * 100:.2f}" for i in range(n)],\n        "u": rng.randn(n)})\n    df["target"] = 3 * pd.to_numeric(df["charges"], errors="coerce").fillna(50) + df["u"] * 5\n    r = M.solve(df, df.drop(columns="target").iloc[:40], target="target", task="regression", time_budget=25)\n    p = r["profile"]\n    assert "charges" in p["numeric_stored_as_text"] and "when" in p["datetime_columns"]\n    assert np.isfinite(r["labels"]).all()\n\n\ndef test_profile_flags_constant_duplicate_id_imbalance():\n    n = 500\n    base = rng.randn(n)\n    df = pd.DataFrame({"id": np.arange(n), "x": base, "x_copy": base, "x_scaled": base * 3 + 1,\n                       "const": 1, "noise": rng.randn(n)})\n    df["target"] = (base + rng.randn(n) * 0.2 > 1.3).astype(int)\n    p = M.solve(df, None, target="target", time_budget=25)["profile"]\n    assert "const" in p["constant_columns"]\n    assert ["x", "x_copy"] in p["duplicate_columns"]\n    assert any(pair[0] == "x" and pair[1] == "x_scaled" for pair in p["perfectly_correlated_columns"])\n    assert "id" in p["id_like_columns"]\n    assert p["is_imbalanced"] is True\n\n\ndef test_tiny_dataset_and_single_class_do_not_crash():\n    tiny = pd.DataFrame({"a": rng.randn(12), "target": [0, 1] * 6})\n    assert len(M.solve(tiny, tiny.drop(columns="target"), target="target", time_budget=10)["labels"]) == 12\n    one = pd.DataFrame({"a": rng.randn(40), "target": ["k"] * 40})\n    r = M.solve(one, one.drop(columns="target"), target="target", time_budget=10)\n    assert set(r["labels"]) == {"k"}\n\n\ndef test_high_cardinality_and_all_missing_column():\n    n = 300\n    df = pd.DataFrame({"city": [f"c{i % 60}" for i in range(n)], "empty": np.nan, "v": rng.randn(n)})\n    df["target"] = df["v"] * 2 + (df["city"].str[1:].astype(int) % 7)\n    r = M.solve(df, df.drop(columns="target"), target="target", task="regression", time_budget=25)\n    assert "empty" in r["profile"]["constant_columns"] and "city" in r["profile"]["high_cardinality_columns"]\n\n\ndef test_clustering_returns_exactly_k_labels_and_handles_missing():\n    from sklearn.datasets import make_blobs\n    X, y = make_blobs(300, 3, centers=3, random_state=1)\n    df = pd.DataFrame(X, columns=list("abc"))\n    df.loc[::17, "a"] = np.nan\n    df["cat"] = rng.choice(["p", "q"], len(df))\n    r = M.solve(df, None, task="clustering", n_clusters=3, time_budget=25)\n    from sklearn.metrics import adjusted_rand_score\n    assert len(set(r["labels"])) == 3 and adjusted_rand_score(y, r["labels"]) > 0.9\n\n\ndef test_cli_end_to_end(tmp_path):\n    df = _clf_frame(200)\n    tr, te = tmp_path / "tr.csv", tmp_path / "te.csv"\n    df.to_csv(tr, index=False)\n    df.drop(columns="target").to_csv(te, index=False)\n    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\n    r = subprocess.run([sys.executable, os.path.join(root, "main.py"), "--train", str(tr), "--test", str(te),\n                        "--target", "target", "--out-dir", str(tmp_path / "o")], capture_output=True, text=True)\n    assert r.returncode == 0, r.stderr\n    assert len(pd.read_csv(tmp_path / "o" / "predictions.csv")) == 200\n    assert "model_report" in json.load(open(tmp_path / "o" / "profile.json"))\n\n\nif __name__ == "__main__":  # works without pytest: python tests/test_pipeline.py\n    import inspect\n    import pathlib\n    import tempfile\n    import traceback\n    failed = 0\n    for name, fn in sorted(globals().items()):\n        if name.startswith("test_") and callable(fn):\n            try:\n                kw = {"tmp_path": pathlib.Path(tempfile.mkdtemp())} if "tmp_path" in inspect.signature(fn).parameters else {}\n                fn(**kw)\n                print("PASS", name)\n            except Exception:\n                failed += 1\n                print("FAIL", name)\n                traceback.print_exc()\n    sys.exit(1 if failed else 0)\n', 'tests/test_fixes.py': '"""Regression checks for the reviewed defects; runnable without pytest."""\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport numpy as np\nimport pandas as pd\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nimport main as M\n\n\ndef test_fresh_budget():\n    old = M.Budget(10)\n    old.started -= 100\n    assert old.left() < 0\n    assert M.Budget(10).left() > 9\n\n\ndef test_constant_clusters_and_ids():\n    df = pd.DataFrame({\'id\': np.arange(101, 131), \'constant\': 1})\n    r = M.solve(df, task=\'clustering\', n_clusters=3, id_col=\'id\')\n    assert len(set(r[\'labels\'])) == 3\n    assert np.array_equal(r[\'ids\'], df.id)\n    assert r[\'id_name\'] == \'id\'\n    assert len(set(M.solve(df, task=\'clustering\', n_clusters=1)[\'labels\'])) == 1\n    try:\n        M.solve(df, task=\'clustering\', n_clusters=31)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\'invalid k must fail\')\n\n\ndef test_missing_rows():\n    df = pd.DataFrame({\'a\':[1.,np.nan,2.,np.nan], \'b\':[\'x\',\'?\',\' \',\'y\']})\n    cleaner = M.FeatureCleaner().fit(df)\n    p = M.build_profile(df, cleaner, None, \'clustering\', 2, None, {}, cleaner.transform(df))\n    assert p[\'rows_with_any_missing\'] == 3\n    assert p[\'total_missing_cells\'] == 4\n\n\ndef test_timeout_checkpoint():\n    # Exercise the actual supervisor with a deliberately stalled worker.\n    root = Path(__file__).resolve().parents[1]\n    with tempfile.TemporaryDirectory() as tmp:\n        folder = Path(tmp)\n        (folder/\'supervisor.py\').write_text((root/\'supervisor.py\').read_text())\n        (folder/\'main.py\').write_text(\'\'\'import sys,json,time\nfrom pathlib import Path\nout=Path(sys.argv[len(sys.argv)-1-sys.argv[::-1].index(\'--out-dir\')+1]); out.mkdir(exist_ok=True)\nc=out/\'checkpoint_1\'; c.mkdir()\n(c/\'predictions.csv\').write_text(\'row_id,prediction\\\\n0,1\\\\n\')\n(c/\'profile.json\').write_text(json.dumps({\'task\':\'classification\',\'model_report\':{\'chosen_models\':{\'prior\':1}}}))\n(out/\'checkpoint.json\').write_text(json.dumps({\'directory\':\'checkpoint_1\'}))\ntime.sleep(60)\n\'\'\')\n        spec = importlib.util.spec_from_file_location(\'test_supervisor\', folder/\'supervisor.py\')\n        sup = importlib.util.module_from_spec(spec); spec.loader.exec_module(sup)\n        started = time.monotonic()\n        assert sup.run([\'--out-dir\',str(folder/\'out\'),\'--time-budget\',\'0.01\']) == 0\n        assert time.monotonic() - started < 12\n        p = json.loads((folder/\'out\'/\'profile.json\').read_text())\n        assert p[\'model_report\'][\'supervisor_timeout\'] is True\n        assert (folder/\'out\'/\'predictions.csv\').read_text().endswith(\'0,1\\n\')\n\n\nif __name__ == \'__main__\':\n    for name, fn in list(globals().items()):\n        if name.startswith(\'test_\'):\n            fn(); print(\'PASS\', name, flush=True)\n', 'tests/test_review_fixes.py': '"""Regression coverage for the four defects found in the submission review.\n\nRun with pytest, or directly with python tests/test_review_fixes.py.\n"""\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\n\nimport numpy as np\nimport pandas as pd\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nimport main as M\n\n\ndef test_sorted_integer_measurements_survive_cleaning():\n    for values in (np.arange(100), np.arange(100)[::-1]):\n        data = pd.DataFrame({"temperature": values, "id": np.arange(100)})\n        cleaner = M.FeatureCleaner().fit(data)\n        cleaned = cleaner.transform(data)\n        assert "temperature" in cleaned, cleaner.dropped_\n        np.testing.assert_array_equal(cleaned["temperature"], values)\n        assert "id" not in cleaned\n\n\ndef test_single_column_csv_preserves_header_and_values():\n    with tempfile.TemporaryDirectory() as tmp:\n        path = Path(tmp) / "one_column.csv"\n        expected = pd.DataFrame({"flag": ["True", "False", "?"]})\n        expected.to_csv(path, index=False)\n        pd.testing.assert_frame_equal(M.read_table(path), expected)\n\n\ndef test_single_column_csv_preserves_quoted_delimiters():\n    with tempfile.TemporaryDirectory() as tmp:\n        path = Path(tmp) / "quoted.csv"\n        expected = pd.DataFrame({"feature": ["a,b", "c,d", "e,f"]})\n        expected.to_csv(path, index=False)\n        pd.testing.assert_frame_equal(M.read_table(path), expected)\n\n\ndef test_supported_delimiters_preserve_table():\n    expected = pd.DataFrame({"feature": [1, 2, 3], "category": ["a", "b", "c"]})\n    with tempfile.TemporaryDirectory() as tmp:\n        for i, sep in enumerate((",", ";", "\\t", "|")):\n            path = Path(tmp) / f"table_{i}.csv"\n            expected.to_csv(path, sep=sep, index=False)\n            pd.testing.assert_frame_equal(M.read_table(path), expected)\n        path = Path(tmp) / "table.tsv"\n        expected.to_csv(path, sep="\\t", index=False)\n        pd.testing.assert_frame_equal(M.read_table(path), expected)\n\n\ndef test_boolean_transform_accepts_strings_and_missing_values():\n    cleaner = M.FeatureCleaner().fit(pd.DataFrame({"flag": [True, False] * 20}))\n    test = pd.DataFrame({"flag": ["True", " false ", "?", "", None,\n                                  True, False, 1, 0, 1.0, 0.0, "YES", "no"]})\n    expected = [1.0, 0.0, np.nan, np.nan, np.nan, 1.0, 0.0, 1.0, 0.0,\n                1.0, 0.0, 1.0, 0.0]\n    np.testing.assert_allclose(cleaner.transform(test)["flag"], expected, equal_nan=True)\n    missing_column = cleaner.transform(pd.DataFrame({"other": [0, 1, 2]}))\n    assert missing_column["flag"].isna().all()\n\n\ndef test_boolean_cli_with_missing_test_markers():\n    with tempfile.TemporaryDirectory() as tmp:\n        folder = Path(tmp)\n        train = pd.DataFrame({"flag": [True, False] * 100,\n                              "noise": np.random.RandomState(2).normal(size=200),\n                              "target": [1, 0] * 100})\n        test = pd.DataFrame({"flag": ["True", "False", "?", ""],\n                             "noise": [0.2, 0.5, -0.1, 0.0]})\n        train.to_csv(folder / "train.csv", index=False)\n        test.to_csv(folder / "test.csv", index=False)\n        result = subprocess.run([\n            sys.executable, str(ROOT / "main.py"), "--train", str(folder / "train.csv"),\n            "--test", str(folder / "test.csv"), "--target", "target",\n            "--task", "classification", "--time-budget", "5", "--out-dir", str(folder / "out"),\n        ], capture_output=True, text=True, timeout=20)\n        assert result.returncode == 0, result.stderr\n        predictions = pd.read_csv(folder / "out" / "predictions.csv")\n        assert len(predictions) == len(test)\n        assert predictions.prediction.iloc[:2].tolist() == [1, 0]\n        assert np.isfinite(predictions.filter(like="proba_").to_numpy()).all()\n\n\ndef test_sorted_predictor_cli_uses_signal_on_unseen_rows():\n    with tempfile.TemporaryDirectory() as tmp:\n        folder = Path(tmp)\n        train_values = np.arange(0, 400, 2)\n        test_values = np.arange(1, 400, 2)\n        pd.DataFrame({"temperature": train_values,\n                      "target": (train_values >= 200).astype(int)}).to_csv(folder / "train.csv", index=False)\n        pd.DataFrame({"temperature": test_values}).to_csv(folder / "test.csv", index=False)\n        result = subprocess.run([\n            sys.executable, str(ROOT / "main.py"), "--train", str(folder / "train.csv"),\n            "--test", str(folder / "test.csv"), "--target", "target",\n            "--task", "classification", "--time-budget", "5", "--out-dir", str(folder / "out"),\n        ], capture_output=True, text=True, timeout=20)\n        assert result.returncode == 0, result.stderr\n        predictions = pd.read_csv(folder / "out" / "predictions.csv")\n        assert len(predictions) == len(test_values)\n        accuracy = np.mean(predictions.prediction.to_numpy() == (test_values >= 200))\n        assert accuracy >= 0.95, accuracy\n        profile = json.loads((folder / "out" / "profile.json").read_text())\n        assert "temperature" not in profile["id_like_columns"]\n        assert "prior" not in profile["model_report"]["chosen_models"]\n\n\ndef test_missing_profile_counts_datetime_markers():\n    n = 100\n    dates = pd.Series(pd.date_range("2020-01-01", periods=n).strftime("%Y-%m-%d"), dtype=object)\n    dates.iloc[3] = "?"\n    dates.iloc[4] = " "\n    numbers = pd.Series([str(i) for i in range(n)], dtype=object)\n    numbers.iloc[3] = "?"\n    numbers.iloc[5] = None\n    flags = pd.Series([True, False] * 50, dtype="boolean")\n    flags.iloc[6] = pd.NA\n    raw = pd.DataFrame({"when": dates, "number": numbers, "flag": flags})\n    cleaner = M.FeatureCleaner().fit(raw)\n    profile = M.build_profile(raw, cleaner, None, "clustering", 2, None, {}, cleaner.transform(raw))\n    assert profile["column_types"]["when"] == "datetime"\n    assert profile["missing_counts"] == {"when": 2, "number": 2, "flag": 1}\n    assert profile["total_missing_cells"] == 5\n    assert profile["rows_with_any_missing"] == 4\n    assert profile["missing_fraction"] == {"when": 0.02, "number": 0.02, "flag": 0.01}\n\n\nif __name__ == "__main__":\n    for name, fn in list(globals().items()):\n        if name.startswith("test_") and callable(fn):\n            fn()\n            print("PASS", name, flush=True)\n', 'validation_results.json': '{\n  "date": "2026-10-24",\n  "python": "3.12.14",\n  "packages": {\n    "numpy": "2.3.5",\n    "pandas": "2.2.3",\n    "scikit-learn": "1.8.0",\n    "scipy": "1.17.0"\n  },\n  "exact_pins_installed": false,\n  "test_count": 19,\n  "tests_passed": 19,\n  "test_scripts": [\n    "tests/test_pipeline.py",\n    "tests/test_fixes.py",\n    "tests/test_review_fixes.py"\n  ],\n  "benchmark_count": 11,\n  "benchmark_scope": "Bundled synthetic examples; not the official nine kickoff datasets.",\n  "wall_time_scope": "Whole CLI process measured by evaluate.py, rounded to 0.1 seconds.",\n  "benchmarks": [\n    {\n      "dataset": "bc",\n      "task": "classification",\n      "wall_seconds": 4.5,\n      "supervisor_timeout": false,\n      "models": {\n        "logreg": 0.5,\n        "hgb_small": 0.5\n      },\n      "accuracy": 0.9517543859649122,\n      "balanced_accuracy": 0.9459415584415585,\n      "macro_f1": 0.948773463510284\n    },\n    {\n      "dataset": "wine",\n      "task": "classification",\n      "wall_seconds": 4.2,\n      "supervisor_timeout": false,\n      "models": {\n        "logreg": 1.0\n      },\n      "accuracy": 0.9444444444444444,\n      "balanced_accuracy": 0.9437012263099219,\n      "macro_f1": 0.9432019492965239\n    },\n    {\n      "dataset": "imb",\n      "task": "classification",\n      "wall_seconds": 7.0,\n      "supervisor_timeout": false,\n      "models": {\n        "hgb_small": 1.0\n      },\n      "accuracy": 0.94,\n      "balanced_accuracy": 0.8457572533365705,\n      "macro_f1": 0.8457572533365705\n    },\n    {\n      "dataset": "moons",\n      "task": "classification",\n      "wall_seconds": 4.9,\n      "supervisor_timeout": false,\n      "models": {\n        "hgb_small": 0.667,\n        "hgb": 0.333\n      },\n      "accuracy": 0.9233333333333333,\n      "balanced_accuracy": 0.9231923192319231,\n      "macro_f1": 0.9232915698546986\n    },\n    {\n      "dataset": "diab",\n      "task": "regression",\n      "wall_seconds": 5.8,\n      "supervisor_timeout": false,\n      "models": {\n        "ridge": 0.5,\n        "hgb": 0.25,\n        "rf": 0.25\n      },\n      "r2": 0.49227895325879134\n    },\n    {\n      "dataset": "fried",\n      "task": "regression",\n      "wall_seconds": 15.9,\n      "supervisor_timeout": false,\n      "models": {\n        "hgb_small": 1.0\n      },\n      "r2": 0.8978035921298735\n    },\n    {\n      "dataset": "bike",\n      "task": "regression",\n      "wall_seconds": 12.1,\n      "supervisor_timeout": false,\n      "models": {\n        "hgb_small": 0.4,\n        "hgb_log": 0.4,\n        "et": 0.2\n      },\n      "r2": 0.9579352262227842\n    },\n    {\n      "dataset": "iris",\n      "task": "clustering",\n      "wall_seconds": 1.7,\n      "supervisor_timeout": false,\n      "models": {\n        "kmeans/num_sel": 1.0\n      },\n      "ari": 0.8856970310281228\n    },\n    {\n      "dataset": "blobs",\n      "task": "clustering",\n      "wall_seconds": 2.2,\n      "supervisor_timeout": false,\n      "models": {\n        "ward/num_sel": 1.0\n      },\n      "ari": 0.9970337326409948\n    },\n    {\n      "dataset": "moons_c",\n      "task": "clustering",\n      "wall_seconds": 1.5,\n      "supervisor_timeout": false,\n      "models": {\n        "gmm_full/num": 1.0\n      },\n      "ari": 0.49639476862220944\n    },\n    {\n      "dataset": "blobs5",\n      "task": "clustering",\n      "wall_seconds": 2.5,\n      "supervisor_timeout": false,\n      "models": {\n        "gmm_spherical/num_sel": 1.0\n      },\n      "ari": 1.0\n    }\n  ],\n  "repository_url": "https://github.com/DimitriAntypas/unknown-dataset-hackathon"\n}\n'}
for name, source in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
ENV = BASE / 'udh_colab_env'
if not (ENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ENV)], check=True)
PYTHON = str(ENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements.txt')], check=True)
print('Setup complete. The notebook kernel is unchanged; models run in an isolated environment.')


In [ ]:
# @title 2. Run a complete demonstration
subprocess.run([PYTHON, 'make_synthetic.py'], cwd=PROJECT, check=True)
DEMO_OUT = PROJECT / 'demo_output'
command = [PYTHON, 'main.py', '--train', 'synth/bc_train.csv', '--test', 'synth/bc_test.csv',
           '--target', 'target', '--task', 'classification', '--id-col', 'row_id', '--out-dir', str(DEMO_OUT)]
result = subprocess.run(command, cwd=PROJECT, capture_output=True, text=True, timeout=60)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr)
print((DEMO_OUT/'predictions.csv').read_text().splitlines()[:6])
print('Demonstration completed. You can now upload your own data below.')


In [ ]:
# @title 3. Optional: run all 19 correctness tests
for script in ('tests/test_pipeline.py', 'tests/test_fixes.py', 'tests/test_review_fixes.py'):
    subprocess.run([PYTHON, script], cwd=PROJECT, check=True, timeout=240)


## Use your own data
Upload **CSV files**, or ZIPs containing CSV files. The file names will be listed below.
For classification/regression, specify the training target column and preferably a separate test CSV without labels.
With no test CSV, predictions are for the training rows: **that is not an independent accuracy evaluation**.
For clustering, choose the number of groups required by your task.

Examples: Telco uses target `Churn` and ID `customerID`; Maintenance uses `Machine failure` and ID `UDI`;
Mall uses clustering and ID `CustomerID`. Five groups for Mall is an example assumption.
For a meaningful Maintenance prediction experiment, exclude `TWF,HDF,PWF,OSF,RNF,Product ID` below;
the failure-type columns expose outcome information. For Bike Sharing count prediction, exclude `casual,registered`.


In [ ]:
# @title 4. Upload CSV files or dataset ZIPs
from google.colab import files as colab_files
DATA = PROJECT / 'data'
DATA.mkdir(exist_ok=True)
uploaded = colab_files.upload()
for name, content in uploaded.items():
    safe_name = Path(name).name
    destination = DATA / safe_name
    destination.write_bytes(content)
    if destination.suffix.lower() == '.zip':
        with zipfile.ZipFile(destination) as archive:
            for entry in archive.infolist():
                if entry.filename.lower().endswith('.csv'):
                    target = DATA / destination.stem / Path(entry.filename).name
                    target.parent.mkdir(exist_ok=True)
                    if target.exists():
                        raise ValueError(f'Duplicate CSV name: {target}. Rename files before uploading.')
                    target.write_bytes(archive.read(entry))
print('Copy the desired relative path into the next cell:')
for path in sorted(DATA.rglob('*.csv')):
    print(path.relative_to(DATA))


In [ ]:
# @title 5. Choose your task and run
train_csv = "" # @param {type:"string"}
test_csv = "" # @param {type:"string"}
task = "classification" # @param ["classification", "regression", "clustering"]
target_column = "" # @param {type:"string"}
id_column = "" # @param {type:"string"}
number_of_clusters = 5 # @param {type:"integer"}
exclude_columns = "" # @param {type:"string"}
time_budget = 40 # @param {type:"integer"}

if not train_csv.strip():
    raise ValueError('Enter a training CSV path from the list in step 4.')
if task != 'clustering' and not target_column.strip():
    raise ValueError('Enter the exact target column name, including capitalization.')
def resolve_csv(name):
    path = (DATA/name.strip()).resolve()
    if not path.is_relative_to(DATA.resolve()) or not path.is_file():
        raise ValueError(f'CSV not found in uploaded data: {name}')
    return path
train_path = resolve_csv(train_csv)
test_path = resolve_csv(test_csv) if test_csv.strip() else None
excluded = [c.strip() for c in exclude_columns.split(',') if c.strip()]
if target_column in excluded or (id_column and id_column in excluded):
    raise ValueError('Do not exclude the target or selected ID column.')
if excluded:
    # Use Python's csv module so this notebook kernel needs no extra packages.
    import csv
    for label, path in [('train', train_path), ('test', test_path)]:
        if path is None:
            continue
        destination = DATA / (label+'_prepared.csv')
        with path.open(newline='', encoding='utf-8-sig') as stream:
            reader = csv.DictReader(stream)
            fields = [c for c in reader.fieldnames if c not in excluded]
            rows = [{c: row[c] for c in fields} for row in reader]
        with destination.open('w', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader(); writer.writerows(rows)
        if label == 'train': train_path = destination
        else: test_path = destination
OUTPUT = Path(tempfile.mkdtemp(prefix='results_', dir=PROJECT))
command = [PYTHON, str(PROJECT/'main.py'), '--train', str(train_path), '--task', task,
           '--out-dir', str(OUTPUT), '--time-budget', str(time_budget)]
if test_path: command += ['--test', str(test_path)]
if task == 'clustering': command += ['--n-clusters', str(number_of_clusters)]
else: command += ['--target', target_column]
if id_column: command += ['--id-col', id_column]
result = subprocess.run(command, cwd=PROJECT, capture_output=True, text=True, timeout=60)
print(result.stdout)
if result.returncode: raise RuntimeError(result.stderr)
profile = json.loads((OUTPUT/'profile.json').read_text())
print('First prediction rows:')
print('\n'.join((OUTPUT/'predictions.csv').read_text().splitlines()[:11]))
print('Model report:', json.dumps(profile['model_report'], indent=2))
if profile['model_report'].get('supervisor_timeout'):
    print('Time limit reached: output uses the last saved checkpoint, which may have lower quality.')


In [ ]:
# @title 6. Download predictions and the data profile
if 'OUTPUT' not in globals() or not (OUTPUT/'profile.json').exists():
    raise RuntimeError('Complete step 5 successfully first.')
RESULT_ZIP = PROJECT / 'hackathon_results.zip'
with zipfile.ZipFile(RESULT_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('predictions.csv', 'profile.json'):
        archive.write(OUTPUT/name, name)
colab_files.download(str(RESULT_ZIP))


## Save and share the notebook
Use **File → Save a copy in Drive** to keep your working copy. Use **Share** if your professor requests a Colab link,
or **File → Download → Download .ipynb** if a notebook upload is requested. Follow the professor's actual submission instructions.
Uploaded datasets and generated outputs are temporary runtime files; download results before disconnecting.

The original project README and validation report are embedded in step 1 and written into `/content/udh_colab`.
The two-moons clustering limitation remains. Run times here do not certify performance on the assessment machine.
